# Phase 4: Feature Engineering

## Project: ExacerbAlert

### Objective
To transform the cleaned sensor datasets into meaningful daily, rolling, patient-specific baseline, data-quality, and cross-sensor features for COPD exacerbation-risk analysis.

### Input Data
Eight cleaned sensor datasets from `data/processed/cleaned/`:
- BRI – Breathing Rate
- EI – Environmental Index
- MI – Mobility Index
- PEI – Pulmonary and Physiological Measurements
- PRVI – PPG-based HRV
- SHRI – Heart Rate Index
- SII – Sound Intensity Index
- SQI – Sleep Quality Index

### Key Principles
1. Never modify the original cleaned datasets.
2. Join datasets using `Participant_ID` and date, never row position.
3. Preserve missing values; never replace missing measurements with zero.
4. Do not blindly forward-fill or interpolate sensor data.
5. Use historical observations only for rolling features and patient-specific baselines.
6. Preserve sensor validity flags and distinguish missing data from invalid measurements.
7. Account for differences in sensor sampling frequencies.
8. Prevent future-data leakage throughout feature engineering.

### Expected Outputs
Feature datasets will be saved in `data/processed/features/`.

## Sampling-Frequency and Data-Coverage Analysis

Before feature generation, all eight cleaned sensor datasets were inspected to understand their sampling patterns and data availability.

The analysis included:

- Dataset dimensions and participant counts.
- Minimum and maximum observation dates.
- Median time intervals between consecutive observations.
- Daily observation counts and participant-level coverage.
- Missing dates and irregular sampling patterns.

The sensors do not share identical sampling frequencies. EI and SII have approximately 10-hour sampling intervals, while BRI, MI, PEI, SHRI and SQI have approximately daily observations. PRVI has irregular observation patterns.

These differences were considered when selecting daily aggregation and historical feature-generation methods.

## Common Participant-Date Timeline

A common daily timeline was created using `Participant_ID` and `Date`.

Key decisions:

- Dates were normalized to UTC.
- A complete participant-date timeline was generated across the observed project date range.
- Sensor datasets were aligned using participant identifiers and dates.
- Missing participant-days were retained rather than discarded.
- Missing measurements were preserved as `NaN`.
- No row-position joins, blind forward filling or zero imputation were used.

This timeline provides a consistent foundation for daily feature integration.

## Daily Sensor Feature Engineering

Daily features were generated independently for BRI, EI, MI, PEI, PRVI, SHRI, SII and SQI.

The aggregation process considered:

- Valid measurement values.
- Existing index and data validity flags.
- Daily means and other appropriate descriptive statistics.
- Valid observation counts.
- Invalid observation counts and data-quality indicators where applicable.

Sensor-specific validity rules were investigated before aggregation. Zero-valued measurements were not automatically treated as missing, and original cleaned datasets were not modified.

PRVI's irregular sampling, PEI's missing physiological measurements and SQI's sparse observations were preserved rather than artificially filled.

## Unified Daily Feature Dataset

The eight daily sensor feature tables were integrated using a one-to-one merge on:

- `Participant_ID`
- `Date`

The resulting dataset retains the common participant-date structure and combines the available daily measurements and validity information from all sensors.

Duplicate participant-date keys and observation preservation were checked after integration.

In [ ]:
# Step 2: Import libraries and configure project paths

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Detect the project root automatically
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Define input and output directories
CLEANED_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"

# Sensor names
SENSORS = ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]

# Display paths and verify input directory
print("Project root:", PROJECT_ROOT.resolve())
print("Cleaned data directory:", CLEANED_DATA_DIR.resolve())
print("Cleaned data directory exists:", CLEANED_DATA_DIR.exists())

print("\nExpected sensor files:")
for sensor in SENSORS:
    file_path = CLEANED_DATA_DIR / f"{sensor}.parquet"
    print(f"{sensor}: {'FOUND' if file_path.exists() else 'MISSING'}")

In [ ]:
# Step 3: Load cleaned sensor datasets

sensor_data = {}

for sensor in SENSORS:
    file_path = CLEANED_DATA_DIR / f"{sensor}.parquet"

    sensor_data[sensor] = pd.read_parquet(file_path)

    print(
        f"{sensor}: "
        f"{sensor_data[sensor].shape[0]:,} rows × "
        f"{sensor_data[sensor].shape[1]} columns"
    )

print("\nAll sensor datasets loaded successfully!")

In [ ]:
import sys
import importlib.util

print("Python executable:")
print(sys.executable)

print("\nPyArrow availability:")
print(importlib.util.find_spec("pyarrow"))

print("\nPandas version:")
print(pd.__version__)

In [ ]:
import pyarrow as pa

print("PyArrow imported successfully!")
print("PyArrow version:", pa.__version__)

# Test Pandas-Parquet compatibility
print("\nTesting Pandas Parquet engine...")

engine = pd.io.parquet.get_engine("pyarrow")

print("Parquet engine:", type(engine).__name__)

In [ ]:
import sys
import pyarrow as pa
import pandas as pd

print("Python executable:", sys.executable)
print("PyArrow version:", pa.__version__)
print("Pandas version:", pd.__version__)

In [ ]:
sensor_data = {}

for sensor in SENSORS:
    file_path = CLEANED_DATA_DIR / f"{sensor}.parquet"

    sensor_data[sensor] = pd.read_parquet(file_path)

    print(
        f"{sensor}: "
        f"{sensor_data[sensor].shape[0]:,} rows × "
        f"{sensor_data[sensor].shape[1]} columns"
    )

print("\nAll sensor datasets loaded successfully!")

In [ ]:
print("DATASET VALIDATION SUMMARY")
print("=" * 60)

for sensor, df in sensor_data.items():
    print(f"\n{sensor}")
    print("-" * 30)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Unique participants:", df["Participant_ID"].nunique())
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Timestamp type:", df["Timestamp"].dtype)

In [ ]:
for sensor in SENSORS:

    df = sensor_data[sensor]

    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{'=' * 55}")
        print(f"{sensor} - MISSING VALUE ANALYSIS")
        print("=" * 55)

        for column, count in missing.items():
            percentage = (count / len(df)) * 100

            print(
                f"{column:<30} "
                f"Missing: {count:>6,} "
                f"({percentage:.2f}%)"
            )

In [ ]:
for sensor in SENSORS:

    df = sensor_data[sensor]

    flag_columns = [
        col for col in ["index_validity", "data_validity"]
        if col in df.columns
    ]

    print(f"\n{'=' * 55}")
    print(f"{sensor} - VALIDITY FLAG ANALYSIS")
    print("=" * 55)

    for col in flag_columns:
        print(f"\n{col} distribution:")
        print(df[col].value_counts(dropna=False).sort_index())

    rows_with_missing = df.isna().any(axis=1).sum()

    print("\nRows containing at least one missing value:", rows_with_missing)
    print("Total rows:", len(df))

In [ ]:
sampling_summary = []

for sensor in SENSORS:

    df = sensor_data[sensor].copy()

    df = df.sort_values(
        ["Participant_ID", "Timestamp"]
    )

    participant_stats = []

    for participant, group in df.groupby("Participant_ID"):

        group = group.sort_values("Timestamp")

        time_diff = (
            group["Timestamp"].diff().dt.total_seconds()
        )

        valid_intervals = time_diff.dropna()

        active_days = group["Timestamp"].dt.floor("D").nunique()

        median_interval = (
            valid_intervals.median()
            if not valid_intervals.empty else np.nan
        )

        participant_stats.append({
            "Participant_ID": participant,
            "Observations": len(group),
            "Active_Days": active_days,
            "Median_Interval_Hours": median_interval / 3600
            if pd.notna(median_interval) else np.nan,
            "Observations_Per_Active_Day": (
                len(group) / active_days if active_days else np.nan
            )
        })

    participant_df = pd.DataFrame(participant_stats)

    sampling_summary.append({
        "Sensor": sensor,
        "Participants": len(participant_df),
        "Total_Observations": len(df),
        "Median_Interval_Hours": (
            participant_df["Median_Interval_Hours"].median()
        ),
        "Min_Participant_Interval_Hours": (
            participant_df["Median_Interval_Hours"].min()
        ),
        "Max_Participant_Interval_Hours": (
            participant_df["Median_Interval_Hours"].max()
        ),
        "Median_Observations_Per_Day": (
            participant_df["Observations_Per_Active_Day"].median()
        )
    })

sampling_summary_df = pd.DataFrame(sampling_summary)

display(
    sampling_summary_df.round(2)
)

In [ ]:
coverage_summary = []
participant_coverage = []

for sensor in SENSORS:

    df = sensor_data[sensor].sort_values(
        ["Participant_ID", "Timestamp"]
    )

    all_gaps = []

    for participant, group in df.groupby("Participant_ID"):

        group = group.sort_values("Timestamp")

        gaps = (
            group["Timestamp"]
            .diff()
            .dt.total_seconds()
            .div(3600)
            .dropna()
        )

        long_gaps = gaps[gaps > 48]

        participant_coverage.append({
            "Sensor": sensor,
            "Participant_ID": participant,
            "Observations": len(group),
            "First_Timestamp": group["Timestamp"].min(),
            "Last_Timestamp": group["Timestamp"].max(),
            "Active_Days": group["Timestamp"].dt.floor("D").nunique(),
            "Max_Gap_Hours": gaps.max() if not gaps.empty else np.nan,
            "Gaps_Over_48h": len(long_gaps)
        })

        all_gaps.extend(gaps.tolist())

    gap_series = pd.Series(all_gaps)

    coverage_summary.append({
        "Sensor": sensor,
        "First_Timestamp": df["Timestamp"].min(),
        "Last_Timestamp": df["Timestamp"].max(),
        "Participants": df["Participant_ID"].nunique(),
        "Total_Gaps": len(gap_series),
        "Gaps_Over_48h": (gap_series > 48).sum(),
        "Gap_95th_Percentile_Hours": (
            gap_series.quantile(0.95) if not gap_series.empty else np.nan
        ),
        "Maximum_Gap_Hours": (
            gap_series.max() if not gap_series.empty else np.nan
        )
    })

coverage_summary_df = pd.DataFrame(coverage_summary)
participant_coverage_df = pd.DataFrame(participant_coverage)

print("SENSOR COVERAGE SUMMARY")
display(coverage_summary_df.round(2))

print("\nPARTICIPANTS WITH LARGEST GAPS")
display(
    participant_coverage_df
    .sort_values("Max_Gap_Hours", ascending=False)
    .groupby("Sensor")
    .head(3)
    .reset_index(drop=True)
)

In [ ]:
daily_coverage = []

for sensor in SENSORS:

    df = sensor_data[sensor].copy()

    # Convert timestamps into UTC calendar dates
    df["Date"] = df["Timestamp"].dt.floor("D")

    for participant, group in df.groupby("Participant_ID"):

        first_date = group["Date"].min()
        last_date = group["Date"].max()

        expected_days = (last_date - first_date).days + 1
        observed_days = group["Date"].nunique()

        coverage_percent = (
            observed_days / expected_days * 100
            if expected_days > 0 else np.nan
        )

        daily_coverage.append({
            "Sensor": sensor,
            "Participant_ID": participant,
            "First_Date": first_date,
            "Last_Date": last_date,
            "Expected_Days": expected_days,
            "Observed_Days": observed_days,
            "Missing_Days": expected_days - observed_days,
            "Coverage_Percent": coverage_percent
        })

daily_coverage_df = pd.DataFrame(daily_coverage)

# Sensor-level coverage summary
sensor_coverage_summary = (
    daily_coverage_df
    .groupby("Sensor")
    .agg(
        Participants=("Participant_ID", "nunique"),
        Median_Coverage=("Coverage_Percent", "median"),
        Mean_Coverage=("Coverage_Percent", "mean"),
        Minimum_Coverage=("Coverage_Percent", "min"),
        Median_Observed_Days=("Observed_Days", "median")
    )
    .reset_index()
)

print("DAILY COVERAGE SUMMARY BY SENSOR")
display(sensor_coverage_summary.round(2))

print("\nFIVE PARTICIPANTS WITH LOWEST COVERAGE PER SENSOR")
display(
    daily_coverage_df
    .sort_values("Coverage_Percent")
    .groupby("Sensor")
    .head(5)
    .reset_index(drop=True)
    .round(2)
)

In [ ]:

# Inspect the overall project date range and participant coverage

all_participants = set().union(
    *[
        set(df["Participant_ID"].dropna().unique())
        for df in sensor_data.values()
    ]
)

print("TOTAL UNIQUE PARTICIPANTS:", len(all_participants))

print("\nOVERALL PROJECT DATE RANGE")

all_timestamps = pd.concat(
    [df["Timestamp"] for df in sensor_data.values()],
    ignore_index=True
)

print("First timestamp:", all_timestamps.min())
print("Last timestamp:", all_timestamps.max())

print("\nPARTICIPANT COVERAGE BY SENSOR")

for sensor in SENSORS:
    participants = set(
        sensor_data[sensor]["Participant_ID"].dropna().unique()
    )

    print(
        f"{sensor}: {len(participants)} participants | "
        f"{len(all_participants - participants)} not present"
    )

print("\nPARTICIPANTS PRESENT IN ALL EIGHT SENSORS")

common_participants = set.intersection(
    *[
        set(df["Participant_ID"].dropna().unique())
        for df in sensor_data.values()
    ]
)

print("Common participants:", len(common_participants))
print("Participants:", sorted(common_participants))


In [ ]:

# Create a complete participant-date timeline

participant_date_ranges = []

for participant in sorted(all_participants):

    participant_timestamps = []

    for sensor in SENSORS:
        df = sensor_data[sensor]

        timestamps = df.loc[
            df["Participant_ID"] == participant,
            "Timestamp"
        ]

        if not timestamps.empty:
            participant_timestamps.extend(timestamps.tolist())

    first_date = min(participant_timestamps).floor("D")
    last_date = max(participant_timestamps).floor("D")

    participant_date_ranges.append({
        "Participant_ID": participant,
        "First_Date": first_date,
        "Last_Date": last_date
    })

participant_ranges_df = pd.DataFrame(participant_date_ranges)

timeline_parts = []

for _, row in participant_ranges_df.iterrows():

    dates = pd.date_range(
        start=row["First_Date"],
        end=row["Last_Date"],
        freq="D",
        tz="UTC"
    )

    timeline_parts.append(
        pd.DataFrame({
            "Participant_ID": row["Participant_ID"],
            "Date": dates
        })
    )

daily_timeline = pd.concat(
    timeline_parts,
    ignore_index=True
)

print("DAILY TIMELINE CREATED")
print("Total participants:", daily_timeline["Participant_ID"].nunique())
print("Total participant-date rows:", len(daily_timeline))
print("Unique participant-date combinations:",
      daily_timeline.drop_duplicates(
          ["Participant_ID", "Date"]
      ).shape[0])

print("\nTIMELINE PREVIEW")
display(daily_timeline.head(10))

print("\nPARTICIPANT DATE RANGE SUMMARY")
display(participant_ranges_df.head(10))


In [ ]:

# Inspect numeric columns available in each sensor dataset

for sensor in SENSORS:

    df = sensor_data[sensor]

    numeric_columns = df.select_dtypes(
        include=["number"]
    ).columns.tolist()

    print(f"\n{'=' * 50}")
    print(f"SENSOR: {sensor}")
    print(f"{'=' * 50}")

    print("Numeric columns:")

    for column in numeric_columns:
        print(
            f"- {column}: "
            f"{df[column].dtype} | "
            f"Missing: {df[column].isna().sum():,}"
        )


In [ ]:

# Step 6B: Create daily BRI features

bri = sensor_data["BRI"].copy()

# Convert timestamps into UTC calendar dates
bri["Date"] = bri["Timestamp"].dt.floor("D")

# Group by participant and date
bri_daily = (
    bri.groupby(["Participant_ID", "Date"])
    .agg(
        BRI_rf_mean_mean=("rf_mean", "mean"),
        BRI_rf_mean_std=("rf_mean", "std"),
        BRI_rf_mean_min=("rf_mean", "min"),
        BRI_rf_mean_max=("rf_mean", "max"),
        BRI_rf_std_mean=("rf_std", "mean"),
        BRI_rf_min_mean=("rf_min", "mean"),
        BRI_rf_max_mean=("rf_max", "mean"),
        BRI_observation_count=("rf_mean", "count"),
        BRI_index_validity=("index_validity", "min"),
        BRI_index_validity_max=("index_validity", "max"),
        BRI_data_validity=("data_validity", "min"),
        BRI_data_validity_max=("data_validity", "max")
    )
    .reset_index()
)

# Identify conflicting validity values within a day
bri_daily["BRI_index_validity_conflict"] = (
    bri_daily["BRI_index_validity"]
    != bri_daily["BRI_index_validity_max"]
)

bri_daily["BRI_data_validity_conflict"] = (
    bri_daily["BRI_data_validity"]
    != bri_daily["BRI_data_validity_max"]
)

# Remove temporary maximum columns after conflict detection
bri_daily = bri_daily.drop(
    columns=[
        "BRI_index_validity_max",
        "BRI_data_validity_max"
    ]
)

print("DAILY BRI FEATURES CREATED")
print("Rows:", len(bri_daily))
print("Participants:", bri_daily["Participant_ID"].nunique())
print("Duplicate participant-date keys:",
      bri_daily.duplicated(["Participant_ID", "Date"]).sum())

print("\nFEATURE PREVIEW")
display(bri_daily.head(10))

print("\nFEATURE DATA TYPES")
display(bri_daily.dtypes)


In [ ]:

# Investigate BRI values against validity flags

bri = sensor_data["BRI"].copy()

print("BRI MEASUREMENT DISTRIBUTION")
display(
    bri[["rf_mean", "rf_std", "rf_min", "rf_max"]]
    .describe()
    .round(3)
)

print("\nRF_MEAN VALUE COUNTS")
display(
    bri["rf_mean"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nBRI INDEX VALIDITY DISTRIBUTION")
display(bri["index_validity"].value_counts(dropna=False))

print("\nBRI DATA VALIDITY DISTRIBUTION")
display(bri["data_validity"].value_counts(dropna=False))

print("\nRF_MEAN BY VALIDITY FLAGS")

display(
    bri.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )
    .agg(
        Total_Rows=("rf_mean", "size"),
        Mean_RF=("rf_mean", "mean"),
        Zero_RF_Count=("rf_mean", lambda x: (x == 0).sum()),
        Positive_RF_Count=("rf_mean", lambda x: (x > 0).sum()),
        Missing_RF_Count=("rf_mean", lambda x: x.isna().sum())
    )
    .reset_index()
    .round(3)
)


In [ ]:

# Rebuild daily BRI features using valid index measurements

bri = sensor_data["BRI"].copy()

bri["Date"] = bri["Timestamp"].dt.floor("D")

# Keep only valid index measurements for measurement aggregation
bri_valid = bri[bri["index_validity"] == 1].copy()

# Aggregate valid measurements
bri_measurements_daily = (
    bri_valid.groupby(["Participant_ID", "Date"])
    .agg(
        BRI_rf_mean_mean=("rf_mean", "mean"),
        BRI_rf_mean_std=("rf_mean", "std"),
        BRI_rf_mean_min=("rf_mean", "min"),
        BRI_rf_mean_max=("rf_mean", "max"),
        BRI_rf_std_mean=("rf_std", "mean"),
        BRI_rf_min_mean=("rf_min", "mean"),
        BRI_rf_max_mean=("rf_max", "mean"),
        BRI_valid_observation_count=("rf_mean", "count")
    )
    .reset_index()
)

# Aggregate quality indicators from all original records
bri_quality_daily = (
    bri.groupby(["Participant_ID", "Date"])
    .agg(
        BRI_observation_count=("Timestamp", "count"),
        BRI_invalid_observation_count=(
            "index_validity",
            lambda x: (x == 0).sum()
        ),
        BRI_index_validity=("index_validity", "min"),
        BRI_index_validity_max=("index_validity", "max"),
        BRI_data_validity=("data_validity", "min"),
        BRI_data_validity_max=("data_validity", "max")
    )
    .reset_index()
)

# Merge the two daily tables using their actual keys
bri_daily = bri_quality_daily.merge(
    bri_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

# Identify validity conflicts
bri_daily["BRI_index_validity_conflict"] = (
    bri_daily["BRI_index_validity"]
    != bri_daily["BRI_index_validity_max"]
)

bri_daily["BRI_data_validity_conflict"] = (
    bri_daily["BRI_data_validity"]
    != bri_daily["BRI_data_validity_max"]
)

bri_daily = bri_daily.drop(
    columns=[
        "BRI_index_validity_max",
        "BRI_data_validity_max"
    ]
)

print("UPDATED DAILY BRI FEATURES")
print("Rows:", len(bri_daily))
print("Valid observations:",
      bri_daily["BRI_valid_observation_count"].sum())
print("Invalid observations:",
      bri_daily["BRI_invalid_observation_count"].sum())
print("Days without valid BRI measurements:",
      bri_daily["BRI_rf_mean_mean"].isna().sum())
print("Duplicate keys:",
      bri_daily.duplicated(["Participant_ID", "Date"]).sum())

print("\nUPDATED FEATURE PREVIEW")
display(bri_daily.head(10))


In [ ]:

# Correct valid observation counts for days without valid measurements

bri_daily["BRI_valid_observation_count"] = (
    bri_daily["BRI_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("FINAL BRI VALIDATION")

print("Total daily records:", len(bri_daily))
print("Valid observations:",
      bri_daily["BRI_valid_observation_count"].sum())
print("Invalid observations:",
      bri_daily["BRI_invalid_observation_count"].sum())

print("Days with valid measurements:",
      (bri_daily["BRI_valid_observation_count"] > 0).sum())

print("Days without valid measurements:",
      (bri_daily["BRI_valid_observation_count"] == 0).sum())

print("Missing valid observation counts:",
      bri_daily["BRI_valid_observation_count"].isna().sum())

print("\nFINAL BRI FEATURE PREVIEW")
display(bri_daily.head(10))


In [ ]:

# Investigate EI measurement validity

ei = sensor_data["EI"].copy()

measurement_columns = [
    "Mean_Temp", "Mean_Hum", "Mean_AQI", "Mean_Lux"
]

print("EI VALIDITY DISTRIBUTION")

print("\nIndex validity:")
display(ei["index_validity"].value_counts(dropna=False))

print("\nData validity:")
display(ei["data_validity"].value_counts(dropna=False))

print("\nMEASUREMENT SUMMARY BY VALIDITY")

display(
    ei.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )
    .agg({
        "Mean_Temp": ["count", "mean", "min", "max"],
        "Mean_Hum": ["mean", "min", "max"],
        "Mean_AQI": ["mean", "min", "max"],
        "Mean_Lux": ["mean", "min", "max"]
    })
    .round(3)
)

print("\nEI ADDITIONAL VALIDITY COLUMN")

print("data_validity_conflict:")
display(ei["data_validity_conflict"].value_counts(dropna=False))


In [ ]:
# Check zero values among valid EI observations

ei_valid = ei[ei["index_validity"] == 1].copy()

measurement_columns = [
    "Mean_Temp",
    "Mean_Hum",
    "Mean_AQI",
    "Mean_Lux"
]

print("VALID EI OBSERVATIONS:", len(ei_valid))

print("\nZERO VALUES IN VALID OBSERVATIONS")

for col in measurement_columns:
    zero_count = (ei_valid[col] == 0).sum()
    print(f"{col}: {zero_count} zero values out of {len(ei_valid)}")

print("\nVALID EI RECORDS WITH AT LEAST ONE ZERO MEASUREMENT")

zero_rows = ei_valid[
    (ei_valid[measurement_columns] == 0).any(axis=1)
]

display(
    zero_rows[
        ["Timestamp", "Participant_ID"] + measurement_columns
    ].head(10)
)

print("\nRECORDS WITH DATA VALIDITY CONFLICT")

display(
    ei[ei["data_validity_conflict"] == True][
        ["Timestamp", "Participant_ID", "index_validity",
         "data_validity"] + measurement_columns
    ].head(10)
)

In [ ]:
# Inspect zero values in temperature, humidity and AQI

core_columns = ["Mean_Temp", "Mean_Hum", "Mean_AQI"]

core_zero_rows = ei_valid[
    (ei_valid[core_columns] == 0).any(axis=1)
].copy()

print("RECORDS WITH ZERO TEMPERATURE, HUMIDITY OR AQI:")
print("Total:", len(core_zero_rows))

display(
    core_zero_rows[
        ["Timestamp", "Participant_ID"] + core_columns + ["Mean_Lux"]
    ].to_string(index=False)
)

print("\nZERO COUNTS BY COLUMN:")
display((core_zero_rows[core_columns] == 0).sum())

In [ ]:
# Create daily EI features

ei = sensor_data["EI"].copy()
ei["Date"] = ei["Timestamp"].dt.floor("D")

# Identify valid measurements
ei_valid = ei[ei["index_validity"] == 1].copy()

measurement_columns = ["Mean_Temp", "Mean_Hum", "Mean_AQI", "Mean_Lux"]

# Identify valid records where all four measurements are zero
ei_valid["EI_all_zero_record"] = (
    ei_valid[measurement_columns].eq(0).all(axis=1)
)

# Add this information to the original EI dataset
ei["EI_all_zero_record"] = False
ei.loc[ei_valid.index, "EI_all_zero_record"] = (
    ei_valid["EI_all_zero_record"]
)

# Keep eligible measurement records
ei_measurements = ei_valid[
    ~ei_valid["EI_all_zero_record"]
].copy()

# Daily measurement aggregation
ei_measurements_daily = (
    ei_measurements.groupby(["Participant_ID", "Date"])
    .agg(
        EI_Temp_mean=("Mean_Temp", "mean"),
        EI_Temp_std_mean=("Std_Temp", "mean"),
        EI_Temp_min=("Min_Temp", "min"),
        EI_Temp_max=("Max_Temp", "max"),

        EI_Hum_mean=("Mean_Hum", "mean"),
        EI_Hum_std_mean=("Std_Hum", "mean"),
        EI_Hum_min=("Min_Hum", "min"),
        EI_Hum_max=("Max_Hum", "max"),

        EI_AQI_mean=("Mean_AQI", "mean"),
        EI_AQI_std_mean=("Std_AQI", "mean"),
        EI_AQI_min=("Min_AQI", "min"),
        EI_AQI_max=("Max_AQI", "max"),

        EI_Lux_mean=("Mean_Lux", "mean"),
        EI_Lux_std_mean=("Std_Lux", "mean"),
        EI_Lux_min=("Min_Lux", "min"),
        EI_Lux_max=("Max_Lux", "max"),

        EI_valid_observation_count=("Mean_Temp", "count")
    )
    .reset_index()
)

# Daily quality aggregation from all original observations
ei_quality_daily = (
    ei.groupby(["Participant_ID", "Date"])
    .agg(
        EI_observation_count=("Timestamp", "count"),
        EI_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),
        EI_invalid_observation_count=(
            "index_validity", lambda x: (x == 0).sum()
        ),
        EI_all_zero_valid_record_count=(
            "EI_all_zero_record", "sum"
        ),
        EI_index_validity_min=("index_validity", "min"),
        EI_index_validity_max=("index_validity", "max"),
        EI_data_validity_min=("data_validity", "min"),
        EI_data_validity_max=("data_validity", "max"),
        EI_data_validity_conflict_count=(
            "data_validity_conflict", "sum"
        )
    )
    .reset_index()
)

# Detect daily validity conflicts
ei_quality_daily["EI_index_validity_conflict"] = (
    ei_quality_daily["EI_index_validity_min"]
    != ei_quality_daily["EI_index_validity_max"]
)

ei_quality_daily["EI_data_validity_conflict"] = (
    ei_quality_daily["EI_data_validity_min"]
    != ei_quality_daily["EI_data_validity_max"]
)

# Combine measurement and quality features
ei_daily = ei_quality_daily.merge(
    ei_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

# Fill only the naturally zero valid-observation count
ei_daily["EI_valid_observation_count"] = (
    ei_daily["EI_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("EI DAILY FEATURES CREATED")
print("Total daily rows:", len(ei_daily))
print("Unique participant-date keys:",
      ei_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0])
print("Total original observations:", ei_daily["EI_observation_count"].sum())
print("Index-valid observations:",
      ei_daily["EI_index_valid_observation_count"].sum())
print("Excluded all-zero valid records:",
      ei_daily["EI_all_zero_valid_record_count"].sum())
print("Eligible measurement observations:",
      ei_daily["EI_valid_observation_count"].sum())
print("Duplicate daily keys:",
      ei_daily.duplicated(["Participant_ID", "Date"]).sum())

display(ei_daily.head(10))

In [ ]:
# Correct the daily data-validity conflict flag

ei_daily["EI_data_validity_conflict"] = (
    ei_daily["EI_data_validity_conflict_count"] > 0
)

print("Original EI conflict records:",
      ei["data_validity_conflict"].sum())

print("Daily participant-dates containing conflicts:",
      ei_daily["EI_data_validity_conflict"].sum())

print("Total conflicts preserved:",
      ei_daily["EI_data_validity_conflict_count"].sum())

display(
    ei_daily.loc[
        ei_daily["EI_data_validity_conflict"],
        [
            "Participant_ID",
            "Date",
            "EI_observation_count",
            "EI_data_validity_conflict_count",
            "EI_data_validity_min",
            "EI_data_validity_max"
        ]
    ]
)

In [ ]:
# Investigate MI measurement validity

mi = sensor_data["MI"].copy()

mi_columns = [
    "gs_mean", "gs_std",
    "sl_mean", "sl_std",
    "smwd_mean", "smwd_std",
    "walked_time", "walked_dist"
]

print("MI VALIDITY DISTRIBUTION")

print("\nIndex validity:")
display(mi["index_validity"].value_counts(dropna=False))

print("\nData validity:")
display(mi["data_validity"].value_counts(dropna=False))

print("\nMEASUREMENT SUMMARY BY VALIDITY")

display(
    mi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[mi_columns]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

print("\nZERO VALUES BY VALIDITY")

display(
    mi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[mi_columns]
    .agg(lambda x: (x == 0).sum())
)

In [ ]:
# Investigate MI records with index_validity=1 and data_validity=0

mi_questionable = mi[
    (mi["index_validity"] == 1) &
    (mi["data_validity"] == 0)
].copy()

print("RECORDS:", len(mi_questionable))

print("\nMEASUREMENT SUMMARY")

display(
    mi_questionable[mi_columns]
    .describe()
    .round(3)
)

print("\nSAMPLE RECORDS")

display(
    mi_questionable[
        ["Timestamp", "Participant_ID"] + mi_columns
    ].head(15)
)

print("\nZERO COUNTS")

display(
    (mi_questionable[mi_columns] == 0).sum()
)

In [ ]:
# Investigate zero values in valid MI observations

mi_valid = mi[
    (mi["index_validity"] == 1) &
    (mi["data_validity"] == 1)
].copy()

print("VALID MI OBSERVATIONS:", len(mi_valid))

print("\nZERO VALUES IN VALID OBSERVATIONS")

display(
    (mi_valid[mi_columns] == 0).sum().to_frame("Zero_Count")
)

print("\nVALID MI MEASUREMENT SUMMARY")

display(
    mi_valid[mi_columns].describe().round(3)
)

print("\nSAMPLE VALID RECORDS")

display(
    mi_valid[
        ["Timestamp", "Participant_ID"] + mi_columns
    ].head(10)
)

In [ ]:
# Create daily MI features

mi = sensor_data["MI"].copy()
mi["Date"] = mi["Timestamp"].dt.floor("D")

mi_columns = [
    "gs_mean", "gs_std",
    "sl_mean", "sl_std",
    "smwd_mean", "smwd_std",
    "walked_time", "walked_dist"
]

# Keep observations where BOTH validity flags are 1
mi_valid = mi[
    (mi["index_validity"] == 1) &
    (mi["data_validity"] == 1)
].copy()

# Daily measurement aggregation
mi_measurements_daily = (
    mi_valid.groupby(["Participant_ID", "Date"])
    .agg(
        MI_gs_mean=("gs_mean", "mean"),
        MI_gs_std_mean=("gs_std", "mean"),
        MI_sl_mean=("sl_mean", "mean"),
        MI_sl_std_mean=("sl_std", "mean"),
        MI_smwd_mean=("smwd_mean", "mean"),
        MI_smwd_std_mean=("smwd_std", "mean"),
        MI_walked_time_mean=("walked_time", "mean"),
        MI_walked_dist_mean=("walked_dist", "mean"),
        MI_valid_observation_count=("Timestamp", "count")
    )
    .reset_index()
)

# Daily quality aggregation using ALL original observations
mi_quality_daily = (
    mi.groupby(["Participant_ID", "Date"])
    .agg(
        MI_observation_count=("Timestamp", "count"),
        MI_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),
        MI_data_valid_observation_count=(
            "data_validity", lambda x: (x == 1).sum()
        ),
        MI_invalid_observation_count=(
            "index_validity",
            lambda x: (
                (x != 1) |
                (mi.loc[x.index, "data_validity"] != 1)
            ).sum()
        ),
        MI_index_validity_min=("index_validity", "min"),
        MI_index_validity_max=("index_validity", "max"),
        MI_data_validity_min=("data_validity", "min"),
        MI_data_validity_max=("data_validity", "max")
    )
    .reset_index()
)

# Detect daily validity conflicts
mi_quality_daily["MI_index_validity_conflict"] = (
    mi_quality_daily["MI_index_validity_min"]
    != mi_quality_daily["MI_index_validity_max"]
)

mi_quality_daily["MI_data_validity_conflict"] = (
    mi_quality_daily["MI_data_validity_min"]
    != mi_quality_daily["MI_data_validity_max"]
)

# Combine measurement and quality features
mi_daily = mi_quality_daily.merge(
    mi_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

mi_daily["MI_valid_observation_count"] = (
    mi_daily["MI_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("MI DAILY FEATURES CREATED")
print("Total daily rows:", len(mi_daily))
print("Unique participant-date keys:",
      mi_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0])
print("Total original observations:", mi_daily["MI_observation_count"].sum())
print("Index-valid observations:",
      mi_daily["MI_index_valid_observation_count"].sum())
print("Data-valid observations:",
      mi_daily["MI_data_valid_observation_count"].sum())
print("Eligible measurement observations:",
      mi_daily["MI_valid_observation_count"].sum())
print("Invalid observations:",
      mi_daily["MI_invalid_observation_count"].sum())
print("Duplicate daily keys:",
      mi_daily.duplicated(["Participant_ID", "Date"]).sum())

display(mi_daily.head(10))

In [ ]:
# Investigate PEI validity and measurement values

pei = sensor_data["PEI"].copy()

pei_columns = [
    "mean_FEV1",
    "range_FEV1_low",
    "range_FEV1_high",
    "mean_PEF",
    "range_PEF_low",
    "range_PEF_high",
    "HR",
    "SpO2"
]

print("PEI VALIDITY DISTRIBUTION")

display(pei["index_validity"].value_counts(dropna=False))
display(pei["data_validity"].value_counts(dropna=False))

print("\nVALIDITY COMBINATIONS")

display(
    pei.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    ).size().to_frame("Observation_Count")
)

print("\nMEASUREMENT SUMMARY BY VALIDITY")

display(
    pei.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[pei_columns]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

print("\nMISSING VALUES BY VALIDITY")

display(
    pei.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[pei_columns]
    .apply(lambda group: group.isna().sum())
)

print("\nZERO VALUES BY VALIDITY")

display(
    pei.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[pei_columns]
    .apply(lambda group: group.eq(0).sum())
)

In [ ]:
# Investigate zero values in valid PEI observations

pei = sensor_data["PEI"].copy()

pei_columns = [
    "mean_FEV1",
    "range_FEV1_low",
    "range_FEV1_high",
    "mean_PEF",
    "range_PEF_low",
    "range_PEF_high",
    "HR",
    "SpO2"
]

pei_valid = pei[
    (pei["index_validity"] == 1) &
    (pei["data_validity"] == 1)
].copy()

print("VALID PEI OBSERVATIONS:", len(pei_valid))

print("\nZERO VALUES IN VALID OBSERVATIONS")

display(
    (pei_valid[pei_columns] == 0)
    .sum()
    .to_frame("Zero_Count")
)

print("\nMISSING VALUES IN VALID OBSERVATIONS")

display(
    pei_valid[pei_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("\nVALID PEI MEASUREMENT SUMMARY")

display(
    pei_valid[pei_columns]
    .describe()
    .round(3)
)

print("\nSAMPLE VALID RECORDS")

display(
    pei_valid[
        ["Timestamp", "Participant_ID"] + pei_columns
    ].head(10)
)

In [ ]:
# Create daily PEI features

pei = sensor_data["PEI"].copy()
pei["Date"] = pei["Timestamp"].dt.floor("D")

pei_columns = [
    "mean_FEV1",
    "range_FEV1_low",
    "range_FEV1_high",
    "mean_PEF",
    "range_PEF_low",
    "range_PEF_high",
    "HR",
    "SpO2"
]

# Only records with both validity flags equal to 1
pei_valid = pei[
    (pei["index_validity"] == 1) &
    (pei["data_validity"] == 1)
].copy()

# Aggregate eligible measurements by participant and date
pei_measurements_daily = (
    pei_valid.groupby(["Participant_ID", "Date"])
    .agg(
        PEI_FEV1_mean=("mean_FEV1", "mean"),
        PEI_FEV1_low_min=("range_FEV1_low", "min"),
        PEI_FEV1_high_max=("range_FEV1_high", "max"),
        PEI_FEV1_observation_count=("mean_FEV1", "count"),

        PEI_PEF_mean=("mean_PEF", "mean"),
        PEI_PEF_low_min=("range_PEF_low", "min"),
        PEI_PEF_high_max=("range_PEF_high", "max"),
        PEI_PEF_observation_count=("mean_PEF", "count"),

        PEI_HR_mean=("HR", "mean"),
        PEI_HR_observation_count=("HR", "count"),

        PEI_SpO2_mean=("SpO2", "mean"),
        PEI_SpO2_observation_count=("SpO2", "count"),

        PEI_valid_observation_count=("Timestamp", "count")
    )
    .reset_index()
)

# Quality aggregation using ALL original observations
pei_quality_daily = (
    pei.groupby(["Participant_ID", "Date"])
    .agg(
        PEI_observation_count=("Timestamp", "count"),

        PEI_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),

        PEI_data_valid_observation_count=(
            "data_validity", lambda x: (x == 1).sum()
        ),

        PEI_invalid_observation_count=(
            "index_validity",
            lambda x: (
                (x != 1) |
                (pei.loc[x.index, "data_validity"] != 1)
            ).sum()
        ),

        PEI_index_validity_min=("index_validity", "min"),
        PEI_index_validity_max=("index_validity", "max"),

        PEI_data_validity_min=("data_validity", "min"),
        PEI_data_validity_max=("data_validity", "max")
    )
    .reset_index()
)

# Detect daily validity conflicts
pei_quality_daily["PEI_index_validity_conflict"] = (
    pei_quality_daily["PEI_index_validity_min"]
    != pei_quality_daily["PEI_index_validity_max"]
)

pei_quality_daily["PEI_data_validity_conflict"] = (
    pei_quality_daily["PEI_data_validity_min"]
    != pei_quality_daily["PEI_data_validity_max"]
)

# Combine measurement and quality features
pei_daily = pei_quality_daily.merge(
    pei_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

# Days without eligible measurements have zero eligible observations
pei_daily["PEI_valid_observation_count"] = (
    pei_daily["PEI_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("PEI DAILY FEATURES CREATED")
print("Total daily rows:", len(pei_daily))
print("Unique participant-date keys:",
      pei_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0])
print("Total original observations:", pei_daily["PEI_observation_count"].sum())
print("Index-valid observations:",
      pei_daily["PEI_index_valid_observation_count"].sum())
print("Data-valid observations:",
      pei_daily["PEI_data_valid_observation_count"].sum())
print("Eligible measurement observations:",
      pei_daily["PEI_valid_observation_count"].sum())
print("Invalid observations:",
      pei_daily["PEI_invalid_observation_count"].sum())
print("Duplicate daily keys:",
      pei_daily.duplicated(["Participant_ID", "Date"]).sum())

display(pei_daily.head(10))

In [ ]:
# Investigate PRVI validity patterns

prvi = sensor_data["PRVI"].copy()

prvi_columns = [
    "mean_HR",
    "range_HR_low",
    "range_HR_high",
    "std_RR",
    "range_stdRR_low",
    "range_stdRR_high",
    "RMSSD",
    "range_RMSSD_low",
    "range_RMSSD_high",
    "nRMSSD",
    "range_nRMSSD_low",
    "range_nRMSSD_high"
]

print("PRVI VALIDITY DISTRIBUTION")

display(prvi["index_validity"].value_counts(dropna=False))
display(prvi["data_validity"].value_counts(dropna=False))

print("\nVALIDITY COMBINATIONS")

display(
    prvi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    ).size().to_frame("Observation_Count")
)

print("\nMISSING VALUES BY VALIDITY")

display(
    prvi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[prvi_columns]
    .agg(lambda x: x.isna().sum())
)

print("\nZERO VALUES BY VALIDITY")

display(
    prvi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[prvi_columns]
    .agg(lambda x: (x == 0).sum())
)

In [ ]:
# Inspect eligible PRVI observations per participant per day

prvi_valid = prvi[
    (prvi["index_validity"] == 1) &
    (prvi["data_validity"] == 1)
].copy()

prvi_valid["Date"] = prvi_valid["Timestamp"].dt.floor("D")

prvi_daily_counts = (
    prvi_valid.groupby(["Participant_ID", "Date"])
    .size()
    .reset_index(name="Valid_Observation_Count")
)

print("PRVI DAILY OBSERVATION FREQUENCY")

display(
    prvi_daily_counts["Valid_Observation_Count"]
    .value_counts()
    .sort_index()
    .to_frame("Number_of_Participant_Days")
)

print("\nDAILY OBSERVATION COUNT SUMMARY")

display(
    prvi_daily_counts["Valid_Observation_Count"]
    .describe()
    .to_frame("Value")
    .round(2)
)

print("\nPARTICIPANTS WITH MOST VALID OBSERVATIONS")

display(
    prvi_daily_counts.groupby("Participant_ID")
    ["Valid_Observation_Count"]
    .agg(["count", "sum", "max"])
    .sort_values("sum", ascending=False)
    .head(10)
)

In [ ]:
# Create daily PRVI features

prvi = sensor_data["PRVI"].copy()
prvi["Date"] = prvi["Timestamp"].dt.floor("D")

# Eligible observations
prvi_valid = prvi[
    (prvi["index_validity"] == 1) &
    (prvi["data_validity"] == 1)
].copy()

# Daily measurement aggregation
prvi_measurements_daily = (
    prvi_valid.groupby(["Participant_ID", "Date"])
    .agg(
        PRVI_HR_mean=("mean_HR", "mean"),
        PRVI_HR_low_min=("range_HR_low", "min"),
        PRVI_HR_high_max=("range_HR_high", "max"),

        PRVI_std_RR_mean=("std_RR", "mean"),
        PRVI_std_RR_low_min=("range_stdRR_low", "min"),
        PRVI_std_RR_high_max=("range_stdRR_high", "max"),

        PRVI_RMSSD_mean=("RMSSD", "mean"),
        PRVI_RMSSD_low_min=("range_RMSSD_low", "min"),
        PRVI_RMSSD_high_max=("range_RMSSD_high", "max"),

        PRVI_nRMSSD_mean=("nRMSSD", "mean"),
        PRVI_nRMSSD_low_min=("range_nRMSSD_low", "min"),
        PRVI_nRMSSD_high_max=("range_nRMSSD_high", "max"),

        PRVI_nRMSSD_observation_count=("nRMSSD", "count"),
        PRVI_valid_observation_count=("Timestamp", "count")
    )
    .reset_index()
)

# Daily quality aggregation using ALL original observations
prvi_quality_daily = (
    prvi.groupby(["Participant_ID", "Date"])
    .agg(
        PRVI_observation_count=("Timestamp", "count"),

        PRVI_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),

        PRVI_missing_index_validity_count=(
            "index_validity", lambda x: x.isna().sum()
        ),

        PRVI_data_valid_observation_count=(
            "data_validity", lambda x: (x == 1).sum()
        )
    )
    .reset_index()
)

# Combine measurement and quality features
prvi_daily = prvi_quality_daily.merge(
    prvi_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

# Days without eligible observations have zero valid count
for column in [
    "PRVI_valid_observation_count",
    "PRVI_nRMSSD_observation_count"
]:
    prvi_daily[column] = (
        prvi_daily[column].fillna(0).astype("int64")
    )

print("PRVI DAILY FEATURES CREATED")
print("Total daily rows:", len(prvi_daily))
print("Unique participant-date keys:",
      prvi_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0])
print("Total original observations:", prvi_daily["PRVI_observation_count"].sum())
print("Index-valid observations:",
      prvi_daily["PRVI_index_valid_observation_count"].sum())
print("Missing index-validity observations:",
      prvi_daily["PRVI_missing_index_validity_count"].sum())
print("Eligible measurement observations:",
      prvi_daily["PRVI_valid_observation_count"].sum())
print("Duplicate daily keys:",
      prvi_daily.duplicated(["Participant_ID", "Date"]).sum())

display(prvi_daily.head(10))

In [ ]:
# Investigate SHRI validity patterns

shri = sensor_data["SHRI"].copy()

shri_columns = [
    "hr_mean",
    "hr_std",
    "hr_max",
    "hr_min"
]

print("SHRI VALIDITY DISTRIBUTION")

display(shri["index_validity"].value_counts(dropna=False))
display(shri["data_validity"].value_counts(dropna=False))

print("\nVALIDITY COMBINATIONS")

display(
    shri.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    ).size().to_frame("Observation_Count")
)

print("\nMEASUREMENT SUMMARY BY VALIDITY")

display(
    shri.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[shri_columns]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

print("\nZERO VALUES BY VALIDITY")

display(
    shri.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[shri_columns]
    .agg(lambda x: (x == 0).sum())
)

In [ ]:
# Create daily SHRI features

shri = sensor_data["SHRI"].copy()
shri["Date"] = shri["Timestamp"].dt.floor("D")

# Eligible observations
shri_valid = shri[
    (shri["index_validity"] == 1) &
    (shri["data_validity"] == 1)
].copy()

# Daily measurement aggregation
shri_measurements_daily = (
    shri_valid.groupby(["Participant_ID", "Date"])
    .agg(
        SHRI_hr_mean=("hr_mean", "mean"),
        SHRI_hr_std_mean=("hr_std", "mean"),
        SHRI_hr_max=("hr_max", "max"),
        SHRI_hr_min=("hr_min", "min"),
        SHRI_valid_observation_count=("Timestamp", "count")
    )
    .reset_index()
)

# Daily quality aggregation using ALL original observations
shri_quality_daily = (
    shri.groupby(["Participant_ID", "Date"])
    .agg(
        SHRI_observation_count=("Timestamp", "count"),

        SHRI_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),

        SHRI_data_valid_observation_count=(
            "data_validity", lambda x: (x == 1).sum()
        ),

        SHRI_invalid_observation_count=(
            "index_validity",
            lambda x: (
                (x != 1) |
                (shri.loc[x.index, "data_validity"] != 1)
            ).sum()
        ),

        SHRI_index_validity_min=("index_validity", "min"),
        SHRI_index_validity_max=("index_validity", "max"),

        SHRI_data_validity_min=("data_validity", "min"),
        SHRI_data_validity_max=("data_validity", "max")
    )
    .reset_index()
)

# Detect daily validity conflicts
shri_quality_daily["SHRI_index_validity_conflict"] = (
    shri_quality_daily["SHRI_index_validity_min"]
    != shri_quality_daily["SHRI_index_validity_max"]
)

shri_quality_daily["SHRI_data_validity_conflict"] = (
    shri_quality_daily["SHRI_data_validity_min"]
    != shri_quality_daily["SHRI_data_validity_max"]
)

# Combine measurement and quality features
shri_daily = shri_quality_daily.merge(
    shri_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

shri_daily["SHRI_valid_observation_count"] = (
    shri_daily["SHRI_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("SHRI DAILY FEATURES CREATED")
print("Total daily rows:", len(shri_daily))
print("Unique participant-date keys:",
      shri_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0])
print("Total original observations:", shri_daily["SHRI_observation_count"].sum())
print("Index-valid observations:",
      shri_daily["SHRI_index_valid_observation_count"].sum())
print("Data-valid observations:",
      shri_daily["SHRI_data_valid_observation_count"].sum())
print("Eligible measurement observations:",
      shri_daily["SHRI_valid_observation_count"].sum())
print("Invalid observations:",
      shri_daily["SHRI_invalid_observation_count"].sum())
print("Duplicate daily keys:",
      shri_daily.duplicated(["Participant_ID", "Date"]).sum())

display(shri_daily.head(10))

In [ ]:
# Investigate SII validity patterns

sii = sensor_data["SII"].copy()

sii_columns = [
    "Mean_SI",
    "Std_SI",
    "Max_SI",
    "Min_SI"
]

print("SII VALIDITY DISTRIBUTION")

display(sii["index_validity"].value_counts(dropna=False))
display(sii["data_validity"].value_counts(dropna=False))

print("\nVALIDITY COMBINATIONS")

display(
    sii.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    ).size().to_frame("Observation_Count")
)

print("\nMEASUREMENT SUMMARY BY VALIDITY")

display(
    sii.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[sii_columns]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

print("\nZERO VALUES BY VALIDITY")

display(
    sii.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[sii_columns]
    .agg(lambda x: (x == 0).sum())
)

In [ ]:
# Inspect zero-valued measurements in eligible SII records

sii_valid = sii[
    (sii["index_validity"] == 1) &
    (sii["data_validity"] == 1)
].copy()

print("VALID SII RECORDS WITH ZERO MEASUREMENTS")

for column in sii_columns:
    zero_rows = sii_valid[sii_valid[column] == 0]

    print(f"\n{column}: {len(zero_rows)} zero records")

    display(
        zero_rows[
            [
                "Participant_ID",
                "Timestamp",
                "Mean_SI",
                "Std_SI",
                "Max_SI",
                "Min_SI",
                "data_validity_conflict"
            ]
        ].head(15)
    )

print("\nDATA VALIDITY CONFLICT DISTRIBUTION")
display(sii["data_validity_conflict"].value_counts(dropna=False))

In [ ]:
# Inspect zero-valued measurements in eligible SII records

sii = sensor_data["SII"].copy()

sii_columns = [
    "Mean_SI",
    "Std_SI",
    "Max_SI",
    "Min_SI"
]

sii_valid = sii[
    (sii["index_validity"] == 1) &
    (sii["data_validity"] == 1)
].copy()

print("VALID SII RECORDS WITH ZERO MEASUREMENTS")

for column in sii_columns:
    zero_rows = sii_valid[sii_valid[column] == 0]

    print(f"\n{column}: {len(zero_rows)} zero records")

    display(
        zero_rows[
            [
                "Participant_ID",
                "Timestamp",
                "Mean_SI",
                "Std_SI",
                "Max_SI",
                "Min_SI",
                "data_validity_conflict"
            ]
        ]
    )

print("\nRECORDS WITH MISSING VALIDITY")

sii_missing_validity = sii[
    sii["index_validity"].isna() |
    sii["data_validity"].isna()
]

display(sii_missing_validity)

In [ ]:
# Create daily SII features

sii = sensor_data["SII"].copy()
sii["Date"] = sii["Timestamp"].dt.floor("D")

# Eligible measurement observations
sii_valid = sii[
    (sii["index_validity"] == 1) &
    (sii["data_validity"] == 1)
].copy()

# Daily measurement aggregation
sii_measurements_daily = (
    sii_valid.groupby(["Participant_ID", "Date"])
    .agg(
        SII_mean=("Mean_SI", "mean"),
        SII_std_mean=("Std_SI", "mean"),
        SII_min=("Min_SI", "min"),
        SII_max=("Max_SI", "max"),
        SII_valid_observation_count=("Timestamp", "count")
    )
    .reset_index()
)

# Daily quality aggregation using ALL original observations
sii_quality_daily = (
    sii.groupby(["Participant_ID", "Date"])
    .agg(
        SII_observation_count=("Timestamp", "count"),

        SII_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),

        SII_data_valid_observation_count=(
            "data_validity", lambda x: (x == 1).sum()
        ),

        SII_missing_data_validity_count=(
            "data_validity", lambda x: x.isna().sum()
        ),

        SII_invalid_observation_count=(
            "index_validity",
            lambda x: (
                (x != 1) |
                (sii.loc[x.index, "data_validity"] != 1)
            ).sum()
        ),

        SII_index_validity_min=("index_validity", "min"),
        SII_index_validity_max=("index_validity", "max"),

        SII_data_validity_min=("data_validity", "min"),
        SII_data_validity_max=("data_validity", "max"),

        SII_data_validity_conflict_count=(
            "data_validity_conflict", "sum"
        )
    )
    .reset_index()
)

# Daily validity conflict flags
sii_quality_daily["SII_index_validity_conflict"] = (
    sii_quality_daily["SII_index_validity_min"]
    != sii_quality_daily["SII_index_validity_max"]
)

sii_quality_daily["SII_data_validity_conflict"] = (
    sii_quality_daily["SII_data_validity_conflict_count"] > 0
)

# Combine measurements and quality features
sii_daily = sii_quality_daily.merge(
    sii_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

sii_daily["SII_valid_observation_count"] = (
    sii_daily["SII_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("SII DAILY FEATURES CREATED")
print("Total daily rows:", len(sii_daily))
print(
    "Unique participant-date keys:",
    sii_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0]
)
print("Total original observations:", sii_daily["SII_observation_count"].sum())
print(
    "Index-valid observations:",
    sii_daily["SII_index_valid_observation_count"].sum()
)
print(
    "Data-valid observations:",
    sii_daily["SII_data_valid_observation_count"].sum()
)
print(
    "Missing data-validity observations:",
    sii_daily["SII_missing_data_validity_count"].sum()
)
print(
    "Eligible measurement observations:",
    sii_daily["SII_valid_observation_count"].sum()
)
print(
    "Invalid observations:",
    sii_daily["SII_invalid_observation_count"].sum()
)
print(
    "Data validity conflicts:",
    sii_daily["SII_data_validity_conflict_count"].sum()
)
print(
    "Duplicate daily keys:",
    sii_daily.duplicated(["Participant_ID", "Date"]).sum()
)

display(sii_daily.head(10))

In [ ]:
# Investigate SQI validity patterns

sqi = sensor_data["SQI"].copy()

sqi_columns = [
    "use_time",
    "num_movements",
    "duration_movements",
    "WASO",
    "sleep_duration",
    "SE"
]

print("SQI VALIDITY DISTRIBUTION")

display(sqi["index_validity"].value_counts(dropna=False))
display(sqi["data_validity"].value_counts(dropna=False))

print("\nVALIDITY COMBINATIONS")

display(
    sqi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    ).size().to_frame("Observation_Count")
)

print("\nMEASUREMENT SUMMARY BY VALIDITY")

display(
    sqi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[sqi_columns]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

print("\nZERO VALUES BY VALIDITY")

display(
    sqi.groupby(
        ["index_validity", "data_validity"],
        dropna=False
    )[sqi_columns]
    .agg(lambda x: (x == 0).sum())
)

In [ ]:
# Inspect eligible SQI observations

sqi_valid = sqi[
    (sqi["index_validity"] == 1) &
    (sqi["data_validity"] == 1)
].copy()

sqi_measurement_columns = [
    "use_time",
    "num_movements",
    "duration_movements",
    "WASO",
    "sleep_duration",
    "SE"
]

print("ELIGIBLE SQI OBSERVATIONS:", len(sqi_valid))

print("\nMISSING VALUES IN ELIGIBLE OBSERVATIONS")

display(
    sqi_valid[sqi_measurement_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("\nZERO VALUES IN ELIGIBLE OBSERVATIONS")

display(
    sqi_valid[sqi_measurement_columns]
    .eq(0)
    .sum()
    .to_frame("Zero_Count")
)

print("\nELIGIBLE SQI MEASUREMENT SUMMARY")

display(
    sqi_valid[sqi_measurement_columns]
    .describe()
    .round(3)
)

print("\nELIGIBLE RECORDS WITH ZERO NUM_MOVEMENTS")

display(
    sqi_valid[sqi_valid["num_movements"] == 0][
        ["Participant_ID", "Timestamp"] + sqi_measurement_columns
    ]
)

print("\nELIGIBLE RECORDS WITH ZERO WASO")

display(
    sqi_valid[sqi_valid["WASO"] == 0][
        ["Participant_ID", "Timestamp"] + sqi_measurement_columns
    ]
)

In [ ]:
# Create daily SQI features

sqi = sensor_data["SQI"].copy()
sqi["Date"] = sqi["Timestamp"].dt.floor("D")

# Eligible measurement observations
sqi_valid = sqi[
    (sqi["index_validity"] == 1) &
    (sqi["data_validity"] == 1)
].copy()

# Daily measurement aggregation
sqi_measurements_daily = (
    sqi_valid.groupby(["Participant_ID", "Date"])
    .agg(
        SQI_num_movements_mean=("num_movements", "mean"),
        SQI_num_movements_max=("num_movements", "max"),
        SQI_WASO_mean=("WASO", "mean"),
        SQI_sleep_duration_mean=("sleep_duration", "mean"),
        SQI_SE_mean=("SE", "mean"),
        SQI_sleep_duration_observation_count=(
            "sleep_duration", "count"
        ),
        SQI_valid_observation_count=("Timestamp", "count")
    )
    .reset_index()
)

# Daily quality aggregation using ALL original observations
sqi_quality_daily = (
    sqi.groupby(["Participant_ID", "Date"])
    .agg(
        SQI_observation_count=("Timestamp", "count"),

        SQI_index_valid_observation_count=(
            "index_validity", lambda x: (x == 1).sum()
        ),

        SQI_data_valid_observation_count=(
            "data_validity", lambda x: (x == 1).sum()
        ),

        SQI_invalid_observation_count=(
            "index_validity",
            lambda x: (
                (x != 1) |
                (sqi.loc[x.index, "data_validity"] != 1)
            ).sum()
        ),

        SQI_index_validity_min=("index_validity", "min"),
        SQI_index_validity_max=("index_validity", "max"),

        SQI_data_validity_min=("data_validity", "min"),
        SQI_data_validity_max=("data_validity", "max")
    )
    .reset_index()
)

# Daily validity conflict flags
sqi_quality_daily["SQI_index_validity_conflict"] = (
    sqi_quality_daily["SQI_index_validity_min"]
    != sqi_quality_daily["SQI_index_validity_max"]
)

sqi_quality_daily["SQI_data_validity_conflict"] = (
    sqi_quality_daily["SQI_data_validity_min"]
    != sqi_quality_daily["SQI_data_validity_max"]
)

# Combine measurements and quality features
sqi_daily = sqi_quality_daily.merge(
    sqi_measurements_daily,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

sqi_daily["SQI_valid_observation_count"] = (
    sqi_daily["SQI_valid_observation_count"]
    .fillna(0)
    .astype("int64")
)

print("SQI DAILY FEATURES CREATED")
print("Total daily rows:", len(sqi_daily))
print(
    "Unique participant-date keys:",
    sqi_daily[["Participant_ID", "Date"]].drop_duplicates().shape[0]
)
print("Total original observations:", sqi_daily["SQI_observation_count"].sum())
print(
    "Index-valid observations:",
    sqi_daily["SQI_index_valid_observation_count"].sum()
)
print(
    "Data-valid observations:",
    sqi_daily["SQI_data_valid_observation_count"].sum()
)
print(
    "Eligible measurement observations:",
    sqi_daily["SQI_valid_observation_count"].sum()
)
print(
    "Invalid observations:",
    sqi_daily["SQI_invalid_observation_count"].sum()
)
print(
    "Duplicate daily keys:",
    sqi_daily.duplicated(["Participant_ID", "Date"]).sum()
)

display(sqi_daily.head(10))

In [ ]:
# Validate all eight daily sensor feature tables

daily_sensor_tables = {
    "BRI": bri_daily,
    "EI": ei_daily,
    "MI": mi_daily,
    "PEI": pei_daily,
    "PRVI": prvi_daily,
    "SHRI": shri_daily,
    "SII": sii_daily,
    "SQI": sqi_daily
}

print("DAILY SENSOR TABLE VALIDATION")

validation_results = []

for sensor, df in daily_sensor_tables.items():

    validation_results.append({
        "Sensor": sensor,
        "Rows": len(df),
        "Unique_Participants": df["Participant_ID"].nunique(),
        "Unique_Dates": df["Date"].nunique(),
        "Duplicate_Keys": df.duplicated(
            ["Participant_ID", "Date"]
        ).sum(),
        "Missing_Participant_ID": df["Participant_ID"].isna().sum(),
        "Missing_Date": df["Date"].isna().sum(),
        "Total_Observations": df[f"{sensor}_observation_count"].sum(),
        "Valid_Observations": df[f"{sensor}_valid_observation_count"].sum()
    })

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

In [ ]:
# Create unified daily feature dataset

final_daily = daily_timeline.copy()

for sensor, df in daily_sensor_tables.items():

    final_daily = final_daily.merge(
        df,
        on=["Participant_ID", "Date"],
        how="left",
        validate="one_to_one"
    )

print("UNIFIED DAILY DATASET CREATED")

print("Total rows:", len(final_daily))

print(
    "Unique participant-date keys:",
    final_daily[["Participant_ID", "Date"]]
    .drop_duplicates()
    .shape[0]
)

print("Unique participants:", final_daily["Participant_ID"].nunique())

print(
    "Duplicate keys:",
    final_daily.duplicated(["Participant_ID", "Date"]).sum()
)

print("Total columns:", len(final_daily.columns))

print("\nDATASET PREVIEW")

display(final_daily.head(10))

In [ ]:
# Validate observation preservation after unified merge

merge_validation = []

for sensor, df in daily_sensor_tables.items():

    observation_column = f"{sensor}_observation_count"
    valid_column = f"{sensor}_valid_observation_count"

    source_observations = len(sensor_data[sensor])

    daily_observations = df[observation_column].sum()

    merged_observations = final_daily[observation_column].sum()

    valid_observations = final_daily[valid_column].sum()

    merge_validation.append({
        "Sensor": sensor,
        "Source_Observations": source_observations,
        "Daily_Observations": daily_observations,
        "Merged_Observations": merged_observations,
        "Eligible_Observations": valid_observations,
        "Missing_Sensor_Days": final_daily[observation_column].isna().sum(),
        "Observation_Count_Match": (
            source_observations == daily_observations == merged_observations
        )
    })

merge_validation_df = pd.DataFrame(merge_validation)

print("MERGE DATA PRESERVATION VALIDATION")

display(merge_validation_df)

print("\nOVERALL CHECKS")

print(
    "Missing Participant IDs:",
    final_daily["Participant_ID"].isna().sum()
)

print(
    "Missing Dates:",
    final_daily["Date"].isna().sum()
)

print(
    "Duplicate Participant-Date Keys:",
    final_daily.duplicated(["Participant_ID", "Date"]).sum()
)

print(
    "All observation counts preserved:",
    merge_validation_df["Observation_Count_Match"].all()
)

In [ ]:
# Check missingness in sensor measurement features

measurement_missingness = []

for sensor in SENSORS:

    measurement_columns = [
        col for col in final_daily.columns
        if col.startswith(f"{sensor}_")
        and not any(
            term in col
            for term in [
                "observation_count",
                "validity",
                "conflict"
            ]
        )
    ]

    for column in measurement_columns:

        available = final_daily[column].notna().sum()
        missing = final_daily[column].isna().sum()

        measurement_missingness.append({
            "Sensor": sensor,
            "Feature": column,
            "Available_Days": available,
            "Missing_Days": missing,
            "Missing_Percentage": round(
                (missing / len(final_daily)) * 100, 2
            )
        })

measurement_missingness_df = pd.DataFrame(measurement_missingness)

print("MEASUREMENT FEATURE MISSINGNESS")

display(
    measurement_missingness_df.sort_values(
        ["Sensor", "Missing_Percentage"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

print("\nSENSOR-LEVEL FEATURE AVAILABILITY")

sensor_missingness_summary = (
    measurement_missingness_df
    .groupby("Sensor")
    .agg(
        Total_Measurement_Features=("Feature", "count"),
        Average_Missing_Percentage=("Missing_Percentage", "mean"),
        Highest_Missing_Percentage=("Missing_Percentage", "max")
    )
    .round(2)
    .reset_index()
)

display(sensor_missingness_summary)

In [ ]:
# Identify features with very low availability

feature_availability = (
    measurement_missingness_df[
        ["Sensor", "Feature", "Available_Days", "Missing_Percentage"]
    ]
    .sort_values(
        ["Available_Days", "Sensor"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("FEATURES WITH ZERO AVAILABLE VALUES")

display(
    feature_availability[
        feature_availability["Available_Days"] == 0
    ]
)

print("\nFEATURES WITH FEWER THAN 100 AVAILABLE DAYS")

display(
    feature_availability[
        (feature_availability["Available_Days"] > 0) &
        (feature_availability["Available_Days"] < 100)
    ]
)

print("\nFEATURE AVAILABILITY SUMMARY")

print("Total measurement features:", len(feature_availability))
print(
    "Features with zero available values:",
    (feature_availability["Available_Days"] == 0).sum()
)
print(
    "Features with fewer than 100 available days:",
    (feature_availability["Available_Days"] < 100).sum()
)

In [ ]:
# Prepare measurement features for rolling calculations

rolling_source_features = (
    measurement_missingness_df.loc[
        measurement_missingness_df["Available_Days"] > 0,
        ["Sensor", "Feature"]
    ]
    .reset_index(drop=True)
)

print("ROLLING FEATURE CANDIDATES")

print("Total candidate features:", len(rolling_source_features))

print("\nFEATURES PER SENSOR")

display(
    rolling_source_features
    .groupby("Sensor")
    .size()
    .reset_index(name="Candidate_Feature_Count")
)

print("\nEXCLUDED FEATURES")

excluded_features = measurement_missingness_df.loc[
    measurement_missingness_df["Available_Days"] == 0,
    ["Sensor", "Feature"]
]

display(excluded_features)

In [ ]:
# Check available measurement days per participant

participant_feature_coverage = []

for _, row in rolling_source_features.iterrows():

    sensor = row["Sensor"]
    feature = row["Feature"]

    available_counts = (
        final_daily.groupby("Participant_ID")[feature]
        .count()
    )

    for participant, count in available_counts.items():

        participant_feature_coverage.append({
            "Sensor": sensor,
            "Feature": feature,
            "Participant_ID": participant,
            "Available_Days": count
        })

participant_feature_coverage_df = pd.DataFrame(
    participant_feature_coverage
)

print("PARTICIPANT-LEVEL FEATURE COVERAGE")

display(
    participant_feature_coverage_df.head(15)
)

print("\nCOVERAGE SUMMARY BY SENSOR")

display(
    participant_feature_coverage_df
    .groupby("Sensor")
    .agg(
        Median_Available_Days=("Available_Days", "median"),
        Mean_Available_Days=("Available_Days", "mean"),
        Minimum_Available_Days=("Available_Days", "min"),
        Maximum_Available_Days=("Available_Days", "max")
    )
    .round(2)
    .reset_index()
)

In [ ]:
# Calculate actual eligible measurement days per participant

sensor_participant_coverage = []

for sensor in SENSORS:

    count_column = f"{sensor}_valid_observation_count"

    for participant, group in final_daily.groupby("Participant_ID"):

        valid_counts = group[count_column].fillna(0)

        available_days = (valid_counts > 0).sum()

        sensor_participant_coverage.append({
            "Sensor": sensor,
            "Participant_ID": participant,
            "Available_Days": available_days,
            "Timeline_Days": len(group),
            "Coverage_Percentage": round(
                available_days / len(group) * 100, 2
            )
        })

sensor_participant_coverage_df = pd.DataFrame(
    sensor_participant_coverage
)

print("PARTICIPANT COVERAGE SUMMARY BY SENSOR")

display(
    sensor_participant_coverage_df
    .groupby("Sensor")
    .agg(
        Participants=("Participant_ID", "nunique"),
        Median_Available_Days=("Available_Days", "median"),
        Mean_Available_Days=("Available_Days", "mean"),
        Participants_With_3_Or_More_Days=(
            "Available_Days", lambda x: (x >= 3).sum()
        ),
        Participants_With_7_Or_More_Days=(
            "Available_Days", lambda x: (x >= 7).sum()
        ),
        Participants_With_14_Or_More_Days=(
            "Available_Days", lambda x: (x >= 14).sum()
        )
    )
    .round(2)
    .reset_index()
)

In [ ]:
# Diagnose historical rolling-window coverage

window_coverage_results = []

for sensor in SENSORS:

    count_column = f"{sensor}_valid_observation_count"

    for participant, group in final_daily.groupby("Participant_ID"):

        group = group.sort_values("Date").copy()

        observed_day = (
            group[count_column].fillna(0) > 0
        ).astype(int)

        for window in [3, 7, 14]:

            historical_count = (
                observed_day.shift(1)
                .rolling(window=window, min_periods=1)
                .sum()
            )

            valid_counts = historical_count.dropna()

            window_coverage_results.append({
                "Sensor": sensor,
                "Participant_ID": participant,
                "Window_Days": window,
                "Median_Observed_Days": valid_counts.median(),
                "Mean_Observed_Days": valid_counts.mean(),
                "Maximum_Observed_Days": valid_counts.max(),
                "Windows_With_At_Least_1_Day": (
                    valid_counts >= 1
                ).sum(),
                "Windows_With_At_Least_2_Days": (
                    valid_counts >= 2
                ).sum()
            })

window_coverage_df = pd.DataFrame(window_coverage_results)

print("HISTORICAL ROLLING-WINDOW COVERAGE")

display(
    window_coverage_df
    .groupby(["Sensor", "Window_Days"])
    .agg(
        Median_Observed_Days=("Median_Observed_Days", "median"),
        Mean_Observed_Days=("Mean_Observed_Days", "mean"),
        Maximum_Observed_Days=("Maximum_Observed_Days", "max"),
        Windows_With_1_Or_More_Days=(
            "Windows_With_At_Least_1_Day", "sum"
        ),
        Windows_With_2_Or_More_Days=(
            "Windows_With_At_Least_2_Days", "sum"
        )
    )
    .round(2)
    .reset_index()
)

In [ ]:
# Validate rolling source measurement columns

rolling_feature_columns = rolling_source_features["Feature"].tolist()

rolling_numeric_validation = []

for feature in rolling_feature_columns:

    values = final_daily[feature]

    rolling_numeric_validation.append({
        "Feature": feature,
        "Data_Type": str(values.dtype),
        "Non_Numeric_Values": (
            values.notna() & ~pd.to_numeric(
                values, errors="coerce"
            ).notna()
        ).sum(),
        "Infinite_Values": np.isinf(
            pd.to_numeric(values, errors="coerce")
        ).sum(),
        "Available_Values": values.notna().sum()
    })

rolling_numeric_validation_df = pd.DataFrame(
    rolling_numeric_validation
)

print("ROLLING SOURCE NUMERIC VALIDATION")

display(rolling_numeric_validation_df)

print("\nVALIDATION SUMMARY")

print(
    "Non-numeric values:",
    rolling_numeric_validation_df["Non_Numeric_Values"].sum()
)

print(
    "Infinite values:",
    rolling_numeric_validation_df["Infinite_Values"].sum()
)

print(
    "All features numeric:",
    rolling_numeric_validation_df["Non_Numeric_Values"].sum() == 0
)

In [ ]:

# Create historical rolling features (optimized version)

rolling_parts = []

for participant, group in final_daily.groupby("Participant_ID", sort=False):

    group = group.sort_values("Date").copy()

    feature_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    for feature in rolling_feature_columns:

        historical_values = group[feature].shift(1)

        for window in [3, 7, 14]:

            rolling_object = historical_values.rolling(
                window=window,
                min_periods=1
            )

            count = rolling_object.count()

            feature_data[
                f"{feature}_rolling_{window}d_mean"
            ] = rolling_object.mean().to_numpy()

            feature_data[
                f"{feature}_rolling_{window}d_std"
            ] = rolling_object.std(ddof=1).where(
                count >= 2
            ).to_numpy()

            feature_data[
                f"{feature}_rolling_{window}d_count"
            ] = count.astype("int64").to_numpy()

    participant_rolling = pd.DataFrame(feature_data)
    rolling_parts.append(participant_rolling)

rolling_features = pd.concat(
    rolling_parts,
    ignore_index=True
)

print("ROLLING FEATURES CREATED")

print("Total rows:", len(rolling_features))
print("Unique participants:", rolling_features["Participant_ID"].nunique())
print(
    "Duplicate keys:",
    rolling_features.duplicated(["Participant_ID", "Date"]).sum()
)
print("Total columns:", len(rolling_features.columns))

display(rolling_features.head(10))


In [ ]:
# Validate rolling features

rolling_value_columns = [
    col for col in rolling_features.columns
    if col.endswith(("_mean", "_std"))
]

rolling_count_columns = [
    col for col in rolling_features.columns
    if col.endswith("_count")
]

# Check infinite values
infinite_values = np.isinf(
    rolling_features[rolling_value_columns].select_dtypes(include=np.number)
).sum().sum()

# Check rolling count limits
count_limit_violations = {}

for window in [3, 7, 14]:
    columns = [
        col for col in rolling_count_columns
        if f"_rolling_{window}d_count" in col
    ]

    count_limit_violations[window] = (
        rolling_features[columns] > window
    ).sum().sum()

print("ROLLING FEATURE VALIDATION")
print("Infinite values:", infinite_values)
print("Count limit violations:", count_limit_violations)
print(
    "Duplicate participant-date keys:",
    rolling_features.duplicated(["Participant_ID", "Date"]).sum()
)

In [ ]:
# Verify that rolling features use only previous dates

test_participant = "GER-001"
test_feature = "MI_gs_mean"

test_df = final_daily.loc[
    final_daily["Participant_ID"] == test_participant,
    ["Participant_ID", "Date", test_feature]
].sort_values("Date").copy()

test_rolling = rolling_features.loc[
    rolling_features["Participant_ID"] == test_participant,
    [
        "Participant_ID",
        "Date",
        f"{test_feature}_rolling_3d_mean",
        f"{test_feature}_rolling_3d_count"
    ]
].sort_values("Date")

leakage_test = test_df.merge(
    test_rolling,
    on=["Participant_ID", "Date"],
    how="inner",
    validate="one_to_one"
)

first_date_check = (
    leakage_test.iloc[0][f"{test_feature}_rolling_3d_count"] == 0
)

print("TEMPORAL LEAKAGE CHECK")
print("Test participant:", test_participant)
print("Test feature:", test_feature)
print("First date has zero historical observations:", first_date_check)

display(leakage_test.head(10))

In [ ]:
# Find available MI features

mi_columns = [
    col for col in final_daily.columns
    if col.startswith("MI_")
]

print("Available MI columns:")
for col in mi_columns:
    print(col)

## Historical Rolling Features

Historical rolling features were generated for 64 eligible daily measurement features across 3-day, 7-day, and 14-day calendar windows.

For each participant and feature, the following statistics were calculated:

- Rolling mean
- Rolling standard deviation
- Rolling observation count

Methodology:
- Rolling calculations were performed separately for each participant.
- `shift(1)` was applied before rolling calculations to exclude the current day's measurement.
- Only previous calendar days were considered.
- Rolling means require at least one valid historical observation.
- Rolling standard deviations require at least two valid historical observations.
- Rolling counts represent the number of available historical measurements in the window.
- Missing measurements were not replaced with zero.
- No future observations were used.

These rolling features describe historical measurement patterns. Low observation counts should be considered when interpreting the resulting statistics.

In [ ]:
# Create patient-specific historical baseline features

baseline_parts = []

for participant, group in final_daily.groupby("Participant_ID", sort=False):

    group = group.sort_values("Date").copy()

    feature_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    for feature in rolling_feature_columns:

        # Exclude the current day's measurement
        historical_values = group[feature].shift(1)

        # Calculate expanding historical statistics
        historical_mean = historical_values.expanding(
            min_periods=1
        ).mean()

        historical_std = historical_values.expanding(
            min_periods=2
        ).std(ddof=1)

        historical_count = historical_values.expanding(
            min_periods=1
        ).count()

        # Require at least 3 previous observations
        feature_data[f"{feature}_baseline_mean"] = (
            historical_mean.where(historical_count >= 3).to_numpy()
        )

        feature_data[f"{feature}_baseline_std"] = (
            historical_std.where(historical_count >= 3).to_numpy()
        )

        feature_data[f"{feature}_baseline_count"] = (
            historical_count.astype("int64").to_numpy()
        )

    baseline_parts.append(pd.DataFrame(feature_data))

baseline_features = pd.concat(
    baseline_parts,
    ignore_index=True
)

print("HISTORICAL BASELINE FEATURES CREATED")
print("Total rows:", len(baseline_features))
print("Unique participants:", baseline_features["Participant_ID"].nunique())
print(
    "Duplicate keys:",
    baseline_features.duplicated(["Participant_ID", "Date"]).sum()
)
print("Total columns:", len(baseline_features.columns))

display(baseline_features.head(10))

In [ ]:
# Validate historical baseline features

baseline_value_columns = [
    col for col in baseline_features.columns
    if col.endswith(("_mean", "_std"))
]

baseline_count_columns = [
    col for col in baseline_features.columns
    if col.endswith("_count")
]

# Check infinite values
infinite_values = np.isinf(
    baseline_features[baseline_value_columns].select_dtypes(
        include=np.number
    )
).sum().sum()

# Check whether baseline counts decrease
count_decreases = 0

for _, group in baseline_features.groupby("Participant_ID"):
    counts = group[baseline_count_columns]
    count_decreases += (counts.diff() < 0).sum().sum()

# Check the minimum observation requirement
threshold_violations = 0

for col in baseline_value_columns:
    feature_prefix = col.rsplit("_baseline_", 1)[0]
    count_col = f"{feature_prefix}_baseline_count"

    threshold_violations += (
        baseline_features[col].notna()
        & (baseline_features[count_col] < 3)
    ).sum()

print("BASELINE FEATURE VALIDATION")
print("Infinite values:", infinite_values)
print("Baseline count decreases:", count_decreases)
print("Minimum observation threshold violations:", threshold_violations)
print(
    "Duplicate participant-date keys:",
    baseline_features.duplicated(["Participant_ID", "Date"]).sum()
)

In [ ]:
# Verify historical baseline excludes current-day measurements

test_participant = "GER-001"
test_feature = "MI_gs_mean"

test_df = final_daily.loc[
    final_daily["Participant_ID"] == test_participant,
    ["Participant_ID", "Date", test_feature]
].sort_values("Date")

test_baseline = baseline_features.loc[
    baseline_features["Participant_ID"] == test_participant,
    [
        "Participant_ID",
        "Date",
        f"{test_feature}_baseline_mean",
        f"{test_feature}_baseline_count"
    ]
].sort_values("Date")

baseline_test = test_df.merge(
    test_baseline,
    on=["Participant_ID", "Date"],
    how="inner",
    validate="one_to_one"
)

print("BASELINE TEMPORAL LEAKAGE CHECK")

display(baseline_test.head(12))

## Patient-Specific Historical Baseline Features

Historical baseline features were calculated separately for each participant and each eligible daily measurement feature.

For every feature, the following statistics were generated:

- Historical mean
- Historical standard deviation
- Historical observation count

Methodology:
- Expanding historical calculations were used instead of fixed rolling windows.
- `shift(1)` was applied to exclude the current day's measurement.
- Only previous observations from the same participant were considered.
- At least 3 previous valid observations were required for baseline mean and standard deviation.
- Historical observation counts were retained even when fewer than 3 observations were available.
- Missing measurements were not replaced with zero.
- No future observations were used.

These baselines represent each participant's historical measurement patterns and can help identify deviations from their own previous measurements.

In [ ]:
# Inspect existing data-quality features

quality_columns = [
    col for col in final_daily.columns
    if any(term in col.lower() for term in [
        "observation_count",
        "validity",
        "conflict"
    ])
]

print("Existing data-quality columns:", len(quality_columns))

for col in quality_columns:
    print(col)

In [ ]:
# Create sensor availability and days-since-last-valid features

quality_parts = []

for participant, group in final_daily.groupby("Participant_ID", sort=False):

    group = group.sort_values("Date").copy()

    quality_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    for sensor in SENSORS:

        valid_count_column = f"{sensor}_valid_observation_count"

        # A sensor is available when at least one valid observation exists
        availability = (
            group[valid_count_column].fillna(0) > 0
        ).astype("int64")

        # Calendar days since the most recent valid observation
        last_valid_date = group["Date"].where(availability == 1).ffill()

        days_since_last = (
            group["Date"] - last_valid_date
        ).dt.days

        quality_data[f"{sensor}_availability"] = (
            availability.to_numpy()
        )

        quality_data[
            f"{sensor}_days_since_last_valid_observation"
        ] = days_since_last.to_numpy()

    quality_parts.append(pd.DataFrame(quality_data))

data_quality_features = pd.concat(
    quality_parts,
    ignore_index=True
)

print("DATA-QUALITY FEATURES CREATED")
print("Total rows:", len(data_quality_features))
print(
    "Unique participants:",
    data_quality_features["Participant_ID"].nunique()
)
print(
    "Duplicate keys:",
    data_quality_features.duplicated(
        ["Participant_ID", "Date"]
    ).sum()
)
print("Total columns:", len(data_quality_features.columns))

display(data_quality_features.head(10))

In [ ]:
# Correct data-quality validation

availability_columns = [
    f"{sensor}_availability" for sensor in SENSORS
]

invalid_availability_values = sum(
    (~data_quality_features[col].isin([0, 1])).sum()
    for col in availability_columns
)

available_day_violations = 0
missing_day_violations = 0
pre_observation_violations = 0

for sensor in SENSORS:

    availability_col = f"{sensor}_availability"
    days_col = f"{sensor}_days_since_last_valid_observation"

    for _, group in data_quality_features.groupby("Participant_ID"):

        group = group.sort_values("Date")

        availability = group[availability_col]
        days_since = group[days_col]

        # Available days must reset to zero
        available_day_violations += (
            (availability == 1) & (days_since != 0)
        ).sum()

        # Missing days after a valid observation should increase by 1
        previous_days = days_since.shift(1)

        missing_after_observation = (
            (availability == 0)
            & previous_days.notna()
        )

        missing_day_violations += (
            missing_after_observation
            & (days_since != previous_days + 1)
        ).sum()

        # Before the first valid observation, days-since must remain NaN
        pre_observation = (
            (availability == 0)
            & previous_days.isna()
        )

        pre_observation_violations += (
            pre_observation & days_since.notna()
        ).sum()

print("CORRECTED DATA-QUALITY VALIDATION")
print("Invalid availability values:", invalid_availability_values)
print("Available-day violations:", available_day_violations)
print("Missing-day counter violations:", missing_day_violations)
print("Pre-observation violations:", pre_observation_violations)

## Data-Quality Features

This section creates daily sensor availability and missing-observation tracking features.

For each sensor:

- **Availability:** 1 if at least one valid observation exists on that participant-date, otherwise 0.
- **Days Since Last Valid Observation:** Number of calendar days since the most recent valid observation.
- The counter resets to 0 on a day with a valid observation.
- Before the first valid observation, the counter remains missing (NaN).

Existing observation counts and validity flags from the unified daily dataset are reused rather than duplicated.

These features preserve missingness and help describe sensor coverage without replacing missing measurements with zero.

In [ ]:
# Inspect measurement features available for cross-sensor engineering

cross_sensor_feature_inventory = measurement_missingness_df[
    ["Sensor", "Feature", "Available_Days", "Missing_Percentage"]
].copy()

display(
    cross_sensor_feature_inventory.sort_values(
        ["Sensor", "Feature"]
    ).reset_index(drop=True)
)

In [ ]:
# Inspect candidate features for cross-sensor engineering

candidate_sensors = ["BRI", "PEI", "PRVI", "SHRI", "SQI"]

candidate_features_df = (
    measurement_missingness_df[
        measurement_missingness_df["Sensor"].isin(candidate_sensors)
    ]
    .sort_values(["Sensor", "Feature"])
    .reset_index(drop=True)
)

display(candidate_features_df.to_string(index=False))

In [ ]:
# Check same-day availability of heart-rate measurements

heart_rate_features = [
    "PEI_HR_mean",
    "PRVI_HR_mean",
    "SHRI_hr_mean"
]

heart_rate_availability = (
    final_daily[["Participant_ID", "Date"] + heart_rate_features]
    .copy()
)

# Convert measurements into availability indicators
for feature in heart_rate_features:
    heart_rate_availability[feature] = (
        heart_rate_availability[feature].notna().astype(int)
    )

# Count same-day availability combinations
heart_rate_availability["Available_Sensor_Count"] = (
    heart_rate_availability[heart_rate_features].sum(axis=1)
)

print("HEART-RATE SENSOR OVERLAP")
print("\nAvailability combinations:")

display(
    heart_rate_availability[
        heart_rate_features + ["Available_Sensor_Count"]
    ].value_counts().reset_index(name="Participant_Days")
)

print("\nDays with at least two sensors available:")
print(
    (heart_rate_availability["Available_Sensor_Count"] >= 2).sum()
)

print("\nDays with all three sensors available:")
print(
    (heart_rate_availability["Available_Sensor_Count"] == 3).sum()
)

In [ ]:
# Explore same-day PEI and PRVI heart-rate differences

pei_prvi_hr = final_daily[
    ["Participant_ID", "Date", "PEI_HR_mean", "PRVI_HR_mean"]
].dropna(
    subset=["PEI_HR_mean", "PRVI_HR_mean"]
).copy()

pei_prvi_hr["PEI_PRVI_HR_difference"] = (
    pei_prvi_hr["PEI_HR_mean"] - pei_prvi_hr["PRVI_HR_mean"]
)

print("PEI-PRVI HEART-RATE COMPARISON")
print("Overlapping participant-days:", len(pei_prvi_hr))
print("Unique participants:", pei_prvi_hr["Participant_ID"].nunique())

print("\nDifference statistics:")
display(
    pei_prvi_hr["PEI_PRVI_HR_difference"].describe().to_frame()
)

print("\nComparison preview:")
display(pei_prvi_hr.head(10))

In [ ]:
# Verify PEI and PRVI heart-rate overlap directly

pei_prvi_overlap_check = final_daily[
    ["PEI_HR_mean", "PRVI_HR_mean"]
].notna()

print("PEI available days:", pei_prvi_overlap_check["PEI_HR_mean"].sum())
print("PRVI available days:", pei_prvi_overlap_check["PRVI_HR_mean"].sum())

print(
    "Both available:",
    (
        pei_prvi_overlap_check["PEI_HR_mean"]
        & pei_prvi_overlap_check["PRVI_HR_mean"]
    ).sum()
)

print(
    "PEI available but PRVI missing:",
    (
        pei_prvi_overlap_check["PEI_HR_mean"]
        & ~pei_prvi_overlap_check["PRVI_HR_mean"]
    ).sum()
)

In [ ]:
# Create cross-sensor feature dataset

cross_sensor_features = final_daily[
    ["Participant_ID", "Date"]
].copy()

# PEI and PRVI heart-rate availability
cross_sensor_features["PEI_PRVI_HR_both_available"] = (
    final_daily["PEI_HR_mean"].notna()
    & final_daily["PRVI_HR_mean"].notna()
).astype("int64")

# Difference only when both measurements are available
cross_sensor_features["PEI_PRVI_HR_difference"] = (
    final_daily["PEI_HR_mean"] - final_daily["PRVI_HR_mean"]
)

print("CROSS-SENSOR FEATURES CREATED")
print("Total rows:", len(cross_sensor_features))
print("Total columns:", len(cross_sensor_features.columns))
print(
    "Days with both measurements:",
    cross_sensor_features["PEI_PRVI_HR_both_available"].sum()
)

display(cross_sensor_features.head(10))

In [ ]:
# Cross-sensor feature validation

duplicate_keys = cross_sensor_features.duplicated(
    subset=["Participant_ID", "Date"]
).sum()

invalid_availability = (
    ~cross_sensor_features["PEI_PRVI_HR_both_available"].isin([0, 1])
).sum()

missing_difference_violations = (
    (cross_sensor_features["PEI_PRVI_HR_both_available"] == 0)
    & cross_sensor_features["PEI_PRVI_HR_difference"].notna()
).sum()

available_difference_violations = (
    (cross_sensor_features["PEI_PRVI_HR_both_available"] == 1)
    & cross_sensor_features["PEI_PRVI_HR_difference"].isna()
).sum()

infinite_values = np.isinf(
    cross_sensor_features["PEI_PRVI_HR_difference"].dropna()
).sum()

print("CROSS-SENSOR VALIDATION")
print("Duplicate participant-date keys:", duplicate_keys)
print("Invalid availability values:", invalid_availability)
print("Missing-difference violations:", missing_difference_violations)
print("Available-difference violations:", available_difference_violations)
print("Infinite difference values:", infinite_values)

## Cross-Sensor Feature Engineering

This section creates exploratory features using measurements from multiple sensors.

### PEI–PRVI Heart-Rate Comparison

Two features are created:

1. **PEI_PRVI_HR_both_available:** Binary indicator showing whether both PEI and PRVI have valid daily heart-rate measurements.
2. **PEI_PRVI_HR_difference:** Difference between PEI and PRVI daily mean heart-rate measurements, calculated as PEI minus PRVI.

The comparison is calculated only when both measurements are available on the same participant-date.

Only 461 participant-days across 49 participants contain both measurements. Therefore, this feature should be treated as an exploratory cross-sensor comparison rather than a clinical or sensor-accuracy measure.

Missing measurements are preserved as NaN. No missing values are replaced with zero.

In [ ]:
# Inspect all feature groups before final integration

feature_groups = {
    "daily": final_daily,
    "rolling": rolling_features,
    "baseline": baseline_features,
    "data_quality": data_quality_features,
    "cross_sensor": cross_sensor_features
}

print("FEATURE GROUP SUMMARY")

for name, df in feature_groups.items():
    print(
        f"{name}: Rows={len(df)}, "
        f"Columns={len(df.columns)}, "
        f"Unique keys={df[['Participant_ID', 'Date']].drop_duplicates().shape[0]}"
    )

print("\nCOLUMN OVERLAPS BETWEEN FEATURE GROUPS")

for name, df in feature_groups.items():
    if name == "daily":
        continue

    overlapping_columns = (
        set(df.columns)
        & set(final_daily.columns)
    ) - {"Participant_ID", "Date"}

    print(f"{name}:", sorted(overlapping_columns))

In [ ]:
# Integrate all engineered feature groups

final_feature_dataset = final_daily.copy()

for name, df in feature_groups.items():

    if name == "daily":
        continue

    final_feature_dataset = final_feature_dataset.merge(
        df,
        on=["Participant_ID", "Date"],
        how="left",
        validate="one_to_one"
    )

print("FINAL FEATURE DATASET CREATED")
print("Total rows:", len(final_feature_dataset))
print("Total columns:", len(final_feature_dataset.columns))
print(
    "Unique participants:",
    final_feature_dataset["Participant_ID"].nunique()
)
print(
    "Duplicate participant-date keys:",
    final_feature_dataset.duplicated(
        subset=["Participant_ID", "Date"]
    ).sum()
)

display(final_feature_dataset.head())

## Feature Integration and Temporal Validation

The daily, rolling, baseline, data-quality and cross-sensor feature groups were integrated into one participant-date dataset.

The final integration uses `Participant_ID` and `Date` as the composite key, with one-to-one merge validation.

The following checks were performed:

- Duplicate participant-date keys.
- Missing and completely empty feature columns.
- Infinite numerical values.
- Feature-group column overlaps.
- Preservation of participant-date records.
- Consistency of feature names across the integrated dataset and metadata.

### Temporal Leakage Checks

Historical calculations were designed to use past observations only.

- Rolling windows use `shift(1)` to exclude the current date.
- Historical baselines use expanding calculations shifted by one observation.
- Data-quality coverage calculations use previous calendar days where specified.
- Cross-sensor comparisons use measurements from the same participant-date.

Independent spot checks were performed for 3-day, 7-day and 14-day rolling calculations and participant-specific historical baselines.

These checks provide evidence that the tested calculations follow the intended temporal rules. They do not constitute an exhaustive verification of every feature value.

## Missingness and Numerical Validation

Feature missingness was examined using descriptive statistics and visualizations.

The analysis included:

- Missing counts and percentages.
- Mean, standard deviation, minimum and maximum.
- Unique-value counts.
- Highly missing features.
- Numerical infinity checks.
- Unusual values requiring further investigation.

Missing measurements were preserved rather than replaced with zero. High missingness was interpreted in the context of sensor availability and historical observation requirements.

Negative SQI sleep-efficiency values were investigated separately and preserved because their cause could not be established from the available data.

## Feature Summary and Dictionary

Two metadata files were generated:

**Feature Summary**

Provides feature-level missingness and descriptive statistics, including data type, missing count, missing percentage, unique values, mean, standard deviation, minimum and maximum.

**Feature Dictionary**

Documents every engineered feature with:

- Feature name.
- Source sensor.
- Feature group.
- Description.
- Formula.
- Units.
- Time window.
- Missing-value handling.
- Leakage considerations.

The dictionary was checked for duplicate feature names and missing documentation fields.

In [ ]:
# Final feature dataset integration validation

expected_rows = len(final_daily)

expected_columns = (
    len(final_daily.columns)
    + len(rolling_features.columns) - 2
    + len(baseline_features.columns) - 2
    + len(data_quality_features.columns) - 2
    + len(cross_sensor_features.columns) - 2
)

missing_participant_ids = final_feature_dataset["Participant_ID"].isna().sum()
missing_dates = final_feature_dataset["Date"].isna().sum()

print("FINAL INTEGRATION VALIDATION")
print("Expected rows:", expected_rows)
print("Actual rows:", len(final_feature_dataset))
print("Expected columns:", expected_columns)
print("Actual columns:", len(final_feature_dataset.columns))
print("Missing participant IDs:", missing_participant_ids)
print("Missing dates:", missing_dates)
print(
    "Duplicate participant-date keys:",
    final_feature_dataset.duplicated(
        subset=["Participant_ID", "Date"]
    ).sum()
)

print("\nKEY MATCHING VALIDATION")

for name, df in feature_groups.items():

    source_keys = df[["Participant_ID", "Date"]].drop_duplicates()

    merged_keys = final_feature_dataset[
        ["Participant_ID", "Date"]
    ].merge(
        source_keys,
        on=["Participant_ID", "Date"],
        how="left",
        indicator=True
    )

    unmatched_keys = (merged_keys["_merge"] != "both").sum()

    print(f"{name} unmatched keys:", unmatched_keys)

In [ ]:
# Final feature dataset missingness summary

final_missingness_df = pd.DataFrame({
    "Feature": final_feature_dataset.columns,
    "Missing_Values": final_feature_dataset.isna().sum().values,
    "Missing_Percentage": (
        final_feature_dataset.isna().mean().values * 100
    )
})

final_missingness_df = (
    final_missingness_df
    .sort_values("Missing_Percentage", ascending=False)
    .reset_index(drop=True)
)

print("FINAL DATASET MISSINGNESS")
print("Total features:", len(final_feature_dataset.columns))
print(
    "Features with missing values:",
    (final_missingness_df["Missing_Values"] > 0).sum()
)
print(
    "Features with no missing values:",
    (final_missingness_df["Missing_Values"] == 0).sum()
)

print("\nTop 20 features by missing percentage:")

display(final_missingness_df.head(20))

In [ ]:
# Identify features with 100% missing values

completely_missing_features = final_missingness_df[
    final_missingness_df["Missing_Percentage"] == 100
].copy()

print("COMPLETELY MISSING FEATURES")
print("Total:", len(completely_missing_features))

display(completely_missing_features)

In [ ]:
# Remove features with 100% missing values

columns_to_remove = completely_missing_features["Feature"].tolist()

final_feature_dataset = final_feature_dataset.drop(
    columns=columns_to_remove
)

print("COMPLETELY MISSING FEATURES REMOVED")
print("Removed columns:", columns_to_remove)
print("Remaining rows:", len(final_feature_dataset))
print("Remaining columns:", len(final_feature_dataset.columns))
print(
    "Remaining completely missing columns:",
    final_feature_dataset.columns[
        final_feature_dataset.isna().all()
    ].tolist()
)

In [ ]:
# Validate numerical features in the final dataset

numeric_columns = final_feature_dataset.select_dtypes(
    include=[np.number]
).columns

non_numeric_measurement_columns = [
    col for col in final_feature_dataset.columns
    if col not in ["Participant_ID", "Date"]
    and col not in numeric_columns
]

infinite_values = np.isinf(
    final_feature_dataset[numeric_columns]
).sum().sum()

print("FINAL NUMERICAL VALIDATION")
print("Total numerical columns:", len(numeric_columns))
print("Non-numeric feature columns:", len(non_numeric_measurement_columns))
print("Total infinite values:", infinite_values)

print("\nNon-numeric feature column names:")
print(non_numeric_measurement_columns)

In [ ]:
# Generate descriptive statistics for numerical features

final_numeric_summary = (
    final_feature_dataset[numeric_columns]
    .describe()
    .T
)

final_numeric_summary["Missing_Percentage"] = (
    final_feature_dataset[numeric_columns].isna().mean() * 100
)

final_numeric_summary["Zero_Count"] = (
    (final_feature_dataset[numeric_columns] == 0).sum()
)

final_numeric_summary["Unique_Values"] = (
    final_feature_dataset[numeric_columns].nunique()
)

print("FINAL NUMERICAL FEATURE SUMMARY")
print("Total numerical features:", len(final_numeric_summary))

display(final_numeric_summary.head(20))

In [ ]:
# Identify potentially unusual numerical features

constant_features = final_numeric_summary[
    final_numeric_summary["Unique_Values"] <= 1
]

negative_minimum_features = final_numeric_summary[
    final_numeric_summary["min"] < 0
]

high_maximum_features = final_numeric_summary[
    (
        final_numeric_summary["max"]
        > final_numeric_summary["75%"] * 5
    )
    & (final_numeric_summary["75%"] > 0)
]

print("CONSTANT NUMERICAL FEATURES")
print("Total:", len(constant_features))
display(constant_features)

print("\nFEATURES WITH NEGATIVE MINIMUM VALUES")
print("Total:", len(negative_minimum_features))
display(negative_minimum_features.head(30))

print("\nFEATURES WITH HIGH MAXIMUM VALUES")
print("Total:", len(high_maximum_features))
display(high_maximum_features.head(30))

In [ ]:
# Investigate negative SQI sleep efficiency values

sqi_source = sensor_data["SQI"]

negative_sqi_se = sqi_source[
    sqi_source["SE"] < 0
][
    [
        "Participant_ID",
        "Timestamp",
        "SE",
        "index_validity",
        "data_validity",
        "sleep_duration",
        "WASO"
    ]
].sort_values("SE")

print("NEGATIVE SQI SE SOURCE RECORDS")
print("Total negative records:", len(negative_sqi_se))

display(negative_sqi_se.head(20))

print("\nNEGATIVE SQI SE SUMMARY")
display(negative_sqi_se["SE"].describe().to_frame())

In [ ]:
# Inspect negative SE records by participant

print("NEGATIVE SE RECORDS BY PARTICIPANT")

display(
    negative_sqi_se.groupby("Participant_ID")
    .agg(
        Negative_Record_Count=("SE", "count"),
        Minimum_SE=("SE", "min"),
        Maximum_SE=("SE", "max")
    )
    .sort_values("Negative_Record_Count", ascending=False)
)

# Inspect nearby observations for the most affected participant
participant_to_inspect = "SPA-019"

negative_dates = negative_sqi_se.loc[
    negative_sqi_se["Participant_ID"] == participant_to_inspect,
    "Timestamp"
]

nearby_records = sqi_source[
    (sqi_source["Participant_ID"] == participant_to_inspect)
    & sqi_source["Timestamp"].between(
        negative_dates.min() - pd.Timedelta(days=2),
        negative_dates.max() + pd.Timedelta(days=2)
    )
][
    [
        "Timestamp",
        "SE",
        "sleep_duration",
        "WASO",
        "index_validity",
        "data_validity"
    ]
].sort_values("Timestamp")

print("\nNEARBY RECORDS FOR SPA-019")
display(nearby_records)

In [ ]:
# Create SQI negative sleep-efficiency anomaly summary

sqi_negative_se_summary = (
    negative_sqi_se.groupby("Participant_ID")
    .agg(
        Negative_SE_Records=("SE", "count"),
        Minimum_SE=("SE", "min"),
        Maximum_SE=("SE", "max")
    )
    .reset_index()
)

total_negative_records = len(negative_sqi_se)

print("SQI DATA-QUALITY INVESTIGATION SUMMARY")
print("Total negative SE records:", total_negative_records)
print("Affected participants:", len(sqi_negative_se_summary))
print(
    "All negative records marked index-valid:",
    (negative_sqi_se["index_validity"] == 1).all()
)
print(
    "All negative records marked data-valid:",
    (negative_sqi_se["data_validity"] == 1).all()
)

display(sqi_negative_se_summary)

### SQI Sleep-Efficiency Data-Quality Investigation

During numerical validation, negative values were identified in the SQI `SE` feature.

Further investigation found:

- Total negative records: 15
- Affected participants: 5
- All 15 records have `index_validity = 1`.
- All 15 records have `data_validity = 1`.
- SPA-019 accounts for 9 of the negative records.

Negative values were also observed across multiple dates for some participants, indicating that the issue is not limited to a single isolated record.

The reason for these negative values could not be established from the available dataset columns.

**Handling decision:**

- Original values are preserved without modification.
- Negative values are not replaced with zero.
- Existing validity flags are retained.
- These records are documented as unresolved data-quality concerns.

Further investigation requires the original SQI sleep-efficiency calculation formula or source documentation.

In [ ]:
# Final temporal consistency validation

final_feature_dataset = final_feature_dataset.sort_values(
    ["Participant_ID", "Date"]
).reset_index(drop=True)

# Check duplicate participant-date combinations
duplicate_keys = final_feature_dataset.duplicated(
    subset=["Participant_ID", "Date"]
).sum()

# Check date continuity for every participant
date_gap_counts = {}

for participant, group in final_feature_dataset.groupby("Participant_ID"):
    date_differences = group["Date"].diff().dropna()

    gaps = (date_differences != pd.Timedelta(days=1)).sum()

    date_gap_counts[participant] = gaps

total_date_gaps = sum(date_gap_counts.values())

# Check chronological ordering
is_chronological = (
    final_feature_dataset.groupby("Participant_ID")["Date"]
    .apply(lambda dates: dates.is_monotonic_increasing)
    .all()
)

print("FINAL TEMPORAL CONSISTENCY VALIDATION")
print("Total rows:", len(final_feature_dataset))
print("Unique participants:", final_feature_dataset["Participant_ID"].nunique())
print("Duplicate participant-date keys:", duplicate_keys)
print("Participants with date gaps:", sum(v > 0 for v in date_gap_counts.values()))
print("Total unexpected date gaps:", total_date_gaps)
print("Dates chronologically ordered:", is_chronological)

In [ ]:
# Validate feature-group integrity

print("FEATURE GROUP INTEGRITY VALIDATION")

for name, df in feature_groups.items():

    duplicate_count = df.duplicated(
        subset=["Participant_ID", "Date"]
    ).sum()

    row_count_match = len(df) == len(final_feature_dataset)

    key_match = (
        set(map(tuple, df[["Participant_ID", "Date"]].values))
        ==
        set(map(tuple, final_feature_dataset[["Participant_ID", "Date"]].values))
    )

    print(f"\n{name.upper()}")
    print("Rows:", len(df))
    print("Duplicate keys:", duplicate_count)
    print("Row count matches:", row_count_match)
    print("Participant-date keys match:", key_match)

In [ ]:
# Validate validity flags and missing-value handling

validity_columns = [
    col for col in final_feature_dataset.columns
    if "validity" in col.lower()
]

print("VALIDITY FLAG VALIDATION")
print("Total validity-related columns:", len(validity_columns))

for col in validity_columns:
    print(f"\n{col}")
    print(final_feature_dataset[col].value_counts(dropna=False).to_dict())

print("\nMISSING-VALUE VALIDATION")
print(
    "Total missing values:",
    final_feature_dataset.isna().sum().sum()
)

print(
    "Completely missing columns:",
    final_feature_dataset.columns[
        final_feature_dataset.isna().all()
    ].tolist()
)

print(
    "Infinite numerical values:",
    np.isinf(
        final_feature_dataset.select_dtypes(include=np.number)
    ).sum().sum()
)

In [ ]:
# Final feature dataset overview

print("FINAL FEATURE DATASET OVERVIEW")
print("-" * 45)

print("Total rows:", final_feature_dataset.shape[0])
print("Total columns:", final_feature_dataset.shape[1])
print("Unique participants:", final_feature_dataset["Participant_ID"].nunique())

print(
    "Date range:",
    final_feature_dataset["Date"].min(),
    "to",
    final_feature_dataset["Date"].max()
)

print(
    "Unique participant-date keys:",
    final_feature_dataset[
        ["Participant_ID", "Date"]
    ].drop_duplicates().shape[0]
)

print(
    "Numeric columns:",
    len(final_feature_dataset.select_dtypes(include=np.number).columns)
)

print(
    "Boolean columns:",
    len(final_feature_dataset.select_dtypes(include="bool").columns)
)

print(
    "Other columns:",
    len(
        final_feature_dataset.select_dtypes(
            exclude=["number", "bool"]
        ).columns
    )
)

print(
    "Total missing values:",
    final_feature_dataset.isna().sum().sum()
)

print("\nFEATURE GROUP SIZES")

for name, df in feature_groups.items():
    feature_count = len(
        [col for col in df.columns if col not in ["Participant_ID", "Date"]]
    )
    print(f"{name}: {feature_count} features")

In [ ]:
# Inspect non-numeric columns in the final dataset

non_numeric_columns = final_feature_dataset.select_dtypes(
    exclude=np.number
).columns

dtype_check = pd.DataFrame({
    "Feature": non_numeric_columns,
    "Data_Type": [
        str(final_feature_dataset[col].dtype)
        for col in non_numeric_columns
    ],
    "Missing_Values": [
        final_feature_dataset[col].isna().sum()
        for col in non_numeric_columns
    ],
    "Unique_Values": [
        final_feature_dataset[col].nunique()
        for col in non_numeric_columns
    ]
})

print("NON-NUMERIC COLUMN INSPECTION")
display(dtype_check)

In [ ]:
# Convert conflict flags to nullable Boolean type

conflict_columns = [
    col for col in final_feature_dataset.columns
    if "conflict" in col.lower()
    and col.endswith("_conflict")
]

for col in conflict_columns:
    final_feature_dataset[col] = (
        final_feature_dataset[col].astype("boolean")
    )

print("CONFLICT FLAG DATA-TYPE CORRECTION")
print("Total conflict columns:", len(conflict_columns))

print("\nCorrected data types:")
print(final_feature_dataset[conflict_columns].dtypes)

print("\nMissing values preserved:")
print(final_feature_dataset[conflict_columns].isna().sum())

In [ ]:
# Recheck final dataset structure after dtype correction

print("UPDATED FINAL DATASET OVERVIEW")
print("Rows:", len(final_feature_dataset))
print("Columns:", len(final_feature_dataset.columns))

print(
    "Numeric columns:",
    len(final_feature_dataset.select_dtypes(include=np.number).columns)
)

print(
    "Boolean columns:",
    len(final_feature_dataset.select_dtypes(include="boolean").columns)
)

print(
    "Other columns:",
    len(
        final_feature_dataset.select_dtypes(
            exclude=["number", "bool", "boolean"]
        ).columns
    )
)

print(
    "Missing values:",
    final_feature_dataset.isna().sum().sum()
)

print(
    "Infinite numerical values:",
    np.isinf(
        final_feature_dataset.select_dtypes(include=np.number)
    ).sum().sum()
)

In [ ]:
# Create final feature summary

feature_summary = pd.DataFrame({
    "Feature": final_feature_dataset.columns,
    "Data_Type": [
        str(final_feature_dataset[col].dtype)
        for col in final_feature_dataset.columns
    ],
    "Missing_Count": [
        final_feature_dataset[col].isna().sum()
        for col in final_feature_dataset.columns
    ],
    "Missing_Percentage": [
        final_feature_dataset[col].isna().mean() * 100
        for col in final_feature_dataset.columns
    ],
    "Unique_Values": [
        final_feature_dataset[col].nunique()
        for col in final_feature_dataset.columns
    ]
})

print("FEATURE SUMMARY CREATED")
print("Total features:", len(feature_summary))

display(feature_summary.head(20))

In [ ]:
# Create the initial feature dictionary

dictionary_rows = []

for feature in final_feature_dataset.columns:

    # Identify source sensor
    source_sensor = next(
        (sensor for sensor in SENSORS if feature.startswith(sensor + "_")),
        "Multiple" if feature.startswith("PEI_PRVI_") else "Metadata"
    )

    # Identify feature group
    if "_rolling_" in feature:
        feature_group = "Rolling"
        time_window = feature.split("_rolling_")[1].split("_")[0] + " days"
        formula = "Statistic calculated using previous calendar days only."
        leakage = "Current-day and future observations excluded."

    elif "_baseline_" in feature:
        feature_group = "Historical Baseline"
        time_window = "Expanding historical"
        formula = "Expanding statistic calculated from previous observations."
        leakage = "Current-day and future observations excluded."

    elif feature.endswith("_availability") or "_days_since_last_valid_observation" in feature:
        feature_group = "Data Quality"
        time_window = "Daily"
        formula = "Calculated from daily valid-observation availability."
        leakage = "Uses current and historical availability information."

    elif feature.startswith("PEI_PRVI_"):
        feature_group = "Cross Sensor"
        time_window = "Daily"
        formula = "Calculated by comparing PEI and PRVI daily heart-rate measurements."
        leakage = "Uses same-day measurements only."

    elif feature in ["Participant_ID", "Date"]:
        feature_group = "Metadata"
        time_window = "Not applicable"
        formula = "Original timeline identifier."
        leakage = "Not applicable."

    else:
        feature_group = "Daily Sensor"
        time_window = "Daily"
        formula = "Daily aggregation or retained data-quality indicator."
        leakage = "Uses observations from the corresponding participant-date."

    dictionary_rows.append({
        "Feature": feature,
        "Source_Sensor": source_sensor,
        "Feature_Group": feature_group,
        "Description": "To be reviewed and finalized.",
        "Formula": formula,
        "Units": "To be confirmed from source documentation.",
        "Time_Window": time_window,
        "Missing_Handling": "Missing values preserved as NaN where applicable.",
        "Leakage_Considerations": leakage
    })

feature_dictionary = pd.DataFrame(dictionary_rows)

print("INITIAL FEATURE DICTIONARY CREATED")
print("Total entries:", len(feature_dictionary))
print("\nFeature groups:")
print(feature_dictionary["Feature_Group"].value_counts())

display(feature_dictionary.head(20))

In [ ]:
# Inspect feature naming patterns

print("ROLLING FEATURE EXAMPLES")
display(pd.Series([
    col for col in rolling_features.columns
    if col not in ["Participant_ID", "Date"]
]).head(15).to_frame("Feature"))

print("\nBASELINE FEATURE EXAMPLES")
display(pd.Series([
    col for col in baseline_features.columns
    if col not in ["Participant_ID", "Date"]
]).head(15).to_frame("Feature"))

print("\nDATA-QUALITY FEATURE NAMES")
display(pd.Series([
    col for col in data_quality_features.columns
    if col not in ["Participant_ID", "Date"]
]).to_frame("Feature"))

print("\nCROSS-SENSOR FEATURE NAMES")
display(pd.Series([
    col for col in cross_sensor_features.columns
    if col not in ["Participant_ID", "Date"]
]).to_frame("Feature"))

In [ ]:
# Improve descriptions and formulas for non-daily feature groups

for index, row in feature_dictionary.iterrows():

    feature = row["Feature"]

    # Metadata
    if feature == "Participant_ID":
        feature_dictionary.loc[index, "Description"] = (
            "Unique identifier assigned to the participant."
        )
        feature_dictionary.loc[index, "Formula"] = "Original participant identifier."
        feature_dictionary.loc[index, "Units"] = "Not applicable"

    elif feature == "Date":
        feature_dictionary.loc[index, "Description"] = (
            "UTC calendar date associated with the participant's daily record."
        )
        feature_dictionary.loc[index, "Formula"] = (
            "Calendar date derived from the observation timestamps."
        )
        feature_dictionary.loc[index, "Units"] = "Date (UTC)"

    # Rolling features
    elif "_rolling_" in feature:

        base_feature, rolling_part = feature.rsplit("_rolling_", 1)
        window, statistic = rolling_part.split("_")

        window_days = int(window.replace("d", ""))

        if statistic == "mean":
            description = (
                f"Mean of available daily {base_feature} values "
                f"over the previous {window_days} calendar days."
            )
            formula = "Mean of non-missing daily values in the previous calendar-day window."
            units = "Same as source feature"

        elif statistic == "std":
            description = (
                f"Standard deviation of available daily {base_feature} values "
                f"over the previous {window_days} calendar days."
            )
            formula = "Standard deviation of non-missing daily values in the previous calendar-day window."
            units = "Same as source feature"

        else:
            description = (
                f"Number of non-missing daily {base_feature} values "
                f"in the previous {window_days} calendar days."
            )
            formula = "Count of non-missing daily values in the previous calendar-day window."
            units = "Observations"

        feature_dictionary.loc[index, "Description"] = description
        feature_dictionary.loc[index, "Formula"] = formula
        feature_dictionary.loc[index, "Units"] = units
        feature_dictionary.loc[index, "Time_Window"] = (
            f"Previous {window_days} calendar days"
        )
        feature_dictionary.loc[index, "Missing_Handling"] = (
            "Missing daily measurements are excluded from the calculation; "
            "minimum observation requirements apply to mean and standard deviation."
        )
        feature_dictionary.loc[index, "Leakage_Considerations"] = (
            "Current-date and future observations are excluded using a one-day shift."
        )

    # Historical baseline features
    elif "_baseline_" in feature:

        base_feature, statistic = feature.rsplit("_baseline_", 1)

        if statistic == "mean":
            description = (
                f"Historical mean of {base_feature} using prior valid daily observations."
            )
            formula = "Expanding mean of prior non-missing daily values."
            units = "Same as source feature"

        elif statistic == "std":
            description = (
                f"Historical standard deviation of {base_feature} "
                "using prior valid daily observations."
            )
            formula = "Expanding standard deviation of prior non-missing daily values."
            units = "Same as source feature"

        else:
            description = (
                f"Number of prior valid daily observations available for {base_feature}."
            )
            formula = "Cumulative count of prior non-missing daily values."
            units = "Observations"

        feature_dictionary.loc[index, "Description"] = description
        feature_dictionary.loc[index, "Formula"] = formula
        feature_dictionary.loc[index, "Units"] = units
        feature_dictionary.loc[index, "Time_Window"] = (
            "Expanding history up to the previous date"
        )
        feature_dictionary.loc[index, "Missing_Handling"] = (
            "Missing daily measurements are excluded; at least 3 prior valid "
            "observations are required for mean and standard deviation."
        )
        feature_dictionary.loc[index, "Leakage_Considerations"] = (
            "Current-date and future observations are excluded using a one-day shift."
        )

    # Data-quality features
    elif feature.endswith("_availability"):
        sensor = feature.replace("_availability", "")

        feature_dictionary.loc[index, "Description"] = (
            f"Indicates whether {sensor} has at least one valid observation "
            "on the participant-date."
        )
        feature_dictionary.loc[index, "Formula"] = (
            "1 if valid observation count is greater than zero; otherwise 0."
        )
        feature_dictionary.loc[index, "Units"] = "Binary (0/1)"
        feature_dictionary.loc[index, "Missing_Handling"] = (
            "No recorded valid observation is represented as 0."
        )
        feature_dictionary.loc[index, "Leakage_Considerations"] = (
            "Uses same-day observation availability."
        )

    elif "_days_since_last_valid_observation" in feature:
        sensor = feature.split("_days_since_last_valid_observation")[0]

        feature_dictionary.loc[index, "Description"] = (
            f"Calendar days since the most recent valid {sensor} observation."
        )
        feature_dictionary.loc[index, "Formula"] = (
            "Current date minus the most recent date with a valid observation."
        )
        feature_dictionary.loc[index, "Units"] = "Days"
        feature_dictionary.loc[index, "Missing_Handling"] = (
            "Remains NaN before the first valid observation."
        )
        feature_dictionary.loc[index, "Leakage_Considerations"] = (
            "Uses current and historical availability; no future observations."
        )

    # Cross-sensor features
    elif feature.startswith("PEI_PRVI_"):

        feature_dictionary.loc[index, "Source_Sensor"] = "Multiple"

        if feature == "PEI_PRVI_HR_both_available":
            feature_dictionary.loc[index, "Description"] = (
                "Indicates whether both PEI and PRVI daily mean heart-rate "
                "measurements are available."
            )
            feature_dictionary.loc[index, "Formula"] = (
                "1 if both PEI_HR_mean and PRVI_HR_mean are available; otherwise 0."
            )
            feature_dictionary.loc[index, "Units"] = "Binary (0/1)"
            feature_dictionary.loc[index, "Missing_Handling"] = (
                "Unavailable measurements result in an availability value of 0."
            )

        elif feature == "PEI_PRVI_HR_difference":
            feature_dictionary.loc[index, "Description"] = (
                "Difference between PEI and PRVI daily mean heart-rate measurements."
            )
            feature_dictionary.loc[index, "Formula"] = (
                "PEI_HR_mean minus PRVI_HR_mean."
            )
            feature_dictionary.loc[index, "Units"] = (
                "Same as heart-rate measurements"
            )
            feature_dictionary.loc[index, "Missing_Handling"] = (
                "NaN when either sensor's daily heart-rate measurement is unavailable."
            )

        feature_dictionary.loc[index, "Leakage_Considerations"] = (
            "Uses same-day measurements only; no future observations."
        )

print("FEATURE DICTIONARY UPDATED")

print("\nUpdated feature-group counts:")
print(feature_dictionary["Feature_Group"].value_counts())

print("\nRolling feature examples:")
display(
    feature_dictionary[
        feature_dictionary["Feature_Group"] == "Rolling"
    ].head(5)
)

print("\nBaseline feature examples:")
display(
    feature_dictionary[
        feature_dictionary["Feature_Group"] == "Historical Baseline"
    ].head(5)
)

print("\nData-quality and cross-sensor features:")
display(
    feature_dictionary[
        feature_dictionary["Feature_Group"].isin(
            ["Data Quality", "Cross Sensor"]
        )
    ]
)

In [ ]:
# Inspect daily feature names for all sensors

for sensor in SENSORS:

    sensor_columns = [
        col for col in final_feature_dataset.columns
        if col.startswith(sensor + "_")
        and "_rolling_" not in col
        and "_baseline_" not in col
        and not col.endswith("_availability")
        and "_days_since_last_valid_observation" not in col
    ]

    print(f"\n{'=' * 50}")
    print(f"{sensor} DAILY FEATURES ({len(sensor_columns)})")
    print("=" * 50)

    display(pd.Series(sensor_columns, name="Feature").to_frame())

In [ ]:
# Correct daily feature documentation

for index, row in feature_dictionary.iterrows():

    if row["Feature_Group"] != "Daily Sensor":
        continue

    feature = row["Feature"]
    sensor = row["Source_Sensor"]

    # Specific observation counts must come before generic counts

    if feature.endswith("_index_valid_observation_count"):
        description = f"Number of {sensor} observations with index_validity equal to 1."
        formula = "Count of source observations where index_validity equals 1."
        units = "Observations"

    elif feature.endswith("_data_valid_observation_count"):
        description = f"Number of {sensor} observations with data_validity equal to 1."
        formula = "Count of source observations where data_validity equals 1."
        units = "Observations"

    elif feature.endswith("_valid_observation_count"):
        description = f"Number of valid {sensor} observations on the participant-date."
        formula = "Count of observations satisfying the sensor-specific validity criteria."
        units = "Observations"

    elif feature.endswith("_invalid_observation_count"):
        description = f"Number of invalid {sensor} observations on the participant-date."
        formula = "Count of observations classified as invalid under the sensor-specific rules."
        units = "Observations"

    elif feature.endswith("_all_zero_valid_record_count"):
        description = "Number of index-valid records whose relevant measurement fields were all zero."
        formula = "Count of index-valid source records with all relevant measurement fields equal to zero."
        units = "Observations"

    elif feature.endswith("_missing_index_validity_count"):
        description = "Number of source observations with missing index_validity."
        formula = "Count of observations where index_validity is missing."
        units = "Observations"

    elif feature.endswith("_missing_data_validity_count"):
        description = "Number of source observations with missing data_validity."
        formula = "Count of observations where data_validity is missing."
        units = "Observations"

    elif feature.endswith("_data_validity_conflict_count"):
        description = "Number of source observations with a data-validity conflict."
        formula = "Count of observations marked with data_validity_conflict."
        units = "Observations"

    elif feature.endswith("_observation_count"):
        description = f"Total number of {sensor} source observations on the participant-date."
        formula = "Count of source observations for the participant-date."
        units = "Observations"

    elif "_validity_min" in feature or "_validity_max" in feature:
        flag = "index_validity" if "index_validity" in feature else "data_validity"
        statistic = "Minimum" if "_min" in feature else "Maximum"

        description = f"{statistic} recorded {flag} value for {sensor} on that date."
        formula = f"{statistic} of available source {flag} values for the participant-date."
        units = "Validity flag"

    elif feature.endswith("_validity_conflict"):
        flag = "index" if "index_validity" in feature else "data"

        description = f"Indicates whether a {flag}-validity conflict was recorded for {sensor}."
        formula = "True if at least one corresponding validity conflict was recorded."
        units = "Boolean"

    elif feature.endswith("_validity"):
        flag = feature.replace(sensor + "_", "")

        description = f"Daily {flag} indicator retained from {sensor} source data."
        formula = "Daily validity indicator derived from the corresponding source validity values."
        units = "Validity flag"

    else:
        # Measurement statistics
        if feature.endswith("_std_mean"):
            description = f"Mean of record-level standard deviations for {feature.replace(sensor + '_', '')}."
            formula = "Mean of available record-level standard deviation values."

        elif feature.endswith("_low_min"):
            description = f"Minimum recorded lower range value for {feature.replace(sensor + '_', '')}."
            formula = "Minimum of available lower-range values."

        elif feature.endswith("_high_max"):
            description = f"Maximum recorded upper range value for {feature.replace(sensor + '_', '')}."
            formula = "Maximum of available upper-range values."

        elif feature.endswith("_mean"):
            description = f"Daily mean of {feature.replace(sensor + '_', '')}."
            formula = "Mean of eligible non-missing measurements for the participant-date."

        elif feature.endswith("_min"):
            description = f"Daily minimum of {feature.replace(sensor + '_', '')}."
            formula = "Minimum of eligible non-missing measurements for the participant-date."

        elif feature.endswith("_max"):
            description = f"Daily maximum of {feature.replace(sensor + '_', '')}."
            formula = "Maximum of eligible non-missing measurements for the participant-date."

        else:
            description = f"Daily aggregated {feature.replace(sensor + '_', '')} feature."
            formula = "Sensor-specific daily aggregation."

        units = "As defined in source dataset; confirmation required"

    feature_dictionary.loc[index, "Description"] = description
    feature_dictionary.loc[index, "Formula"] = formula
    feature_dictionary.loc[index, "Units"] = units

print("DAILY FEATURE DOCUMENTATION CORRECTED")

check_features = [
    "BRI_observation_count",
    "BRI_invalid_observation_count",
    "BRI_valid_observation_count",
    "EI_index_valid_observation_count",
    "MI_data_valid_observation_count",
    "PRVI_missing_index_validity_count",
    "PEI_FEV1_low_min",
    "PEI_FEV1_high_max",
    "BRI_index_validity"
]

display(
    feature_dictionary[
        feature_dictionary["Feature"].isin(check_features)
    ][["Feature", "Description", "Formula", "Units"]]
)

In [ ]:
# Add sensor-specific documentation notes

special_notes = {
    "BRI": (
        "BRI generally has one eligible observation per day. "
        "Daily standard deviation across observations may therefore be unavailable."
    ),
    "EI": (
        "Multiple observations may occur per day. Daily statistics are calculated "
        "from eligible records; validity conflicts are preserved."
    ),
    "MI": (
        "Daily features use observations satisfying both index_validity and data_validity."
    ),
    "PEI": (
        "FEV1, PEF, HR and SpO2 are aggregated independently. Missing measurement "
        "fields remain missing and do not invalidate other available fields."
    ),
    "PRVI": (
        "PRVI has irregular sampling and incomplete participant coverage. "
        "Daily aggregation uses available eligible observations only."
    ),
    "SHRI": (
        "Daily measurement statistics use observations satisfying both validity conditions."
    ),
    "SII": (
        "Multiple observations may occur per day. Validity conflicts are preserved "
        "and eligible measurement values are retained."
    ),
    "SQI": (
        "SQI measurements are sparse. Missing fields remain missing, and negative "
        "sleep-efficiency values identified during quality investigation are preserved."
    )
}

for index, row in feature_dictionary.iterrows():

    if row["Feature_Group"] != "Daily Sensor":
        continue

    sensor = row["Source_Sensor"]

    feature_dictionary.loc[index, "Missing_Handling"] += (
        " " + special_notes[sensor]
    )

# Clarify PEI range features
range_descriptions = {
    "PEI_FEV1_low_min": (
        "Minimum of the recorded lower FEV1 range values for the participant-date."
    ),
    "PEI_FEV1_high_max": (
        "Maximum of the recorded upper FEV1 range values for the participant-date."
    ),
    "PEI_PEF_low_min": (
        "Minimum of the recorded lower PEF range values for the participant-date."
    ),
    "PEI_PEF_high_max": (
        "Maximum of the recorded upper PEF range values for the participant-date."
    )
}

for feature, description in range_descriptions.items():
    feature_dictionary.loc[
        feature_dictionary["Feature"] == feature, "Description"
    ] = description

print("SENSOR-SPECIFIC DOCUMENTATION ADDED")

print("\nPRVI examples:")
display(
    feature_dictionary[
        (feature_dictionary["Source_Sensor"] == "PRVI")
        & (feature_dictionary["Feature_Group"] == "Daily Sensor")
    ][["Feature", "Description", "Missing_Handling"]].head(5)
)

print("\nPEI range examples:")
display(
    feature_dictionary[
        feature_dictionary["Feature"].isin(range_descriptions.keys())
    ][["Feature", "Description", "Formula"]]
)

print("\nSQI examples:")
display(
    feature_dictionary[
        (feature_dictionary["Source_Sensor"] == "SQI")
        & (feature_dictionary["Feature_Group"] == "Daily Sensor")
    ][["Feature", "Missing_Handling"]].tail(4)
)

In [ ]:
# Final feature dictionary validation

print("FEATURE DICTIONARY VALIDATION")

print("\nTotal entries:", len(feature_dictionary))
print("Duplicate feature names:", feature_dictionary["Feature"].duplicated().sum())

print("\nMissing values by column:")
display(feature_dictionary.isna().sum().to_frame("Missing_Count"))

print("\nRemaining description placeholders:",
      feature_dictionary["Description"].str.contains(
          "To be reviewed|To be finalized",
          case=False,
          na=False
      ).sum())

print("\nRemaining unit placeholders:",
      feature_dictionary["Units"].str.contains(
          "To be confirmed",
          case=False,
          na=False
      ).sum())

print("\nFeature group distribution:")
display(feature_dictionary["Feature_Group"].value_counts().to_frame("Count"))

print("\nSource sensor distribution:")
display(feature_dictionary["Source_Sensor"].value_counts().to_frame("Count"))

print("\nFeatures with empty descriptions:")
display(
    feature_dictionary.loc[
        feature_dictionary["Description"].isna()
        | (feature_dictionary["Description"].str.strip() == ""),
        ["Feature", "Feature_Group"]
    ]
)

print("\nFeatures with empty formulas:")
display(
    feature_dictionary.loc[
        feature_dictionary["Formula"].isna()
        | (feature_dictionary["Formula"].str.strip() == ""),
        ["Feature", "Feature_Group"]
    ]
)

In [ ]:
# Enhance feature summary with descriptive statistics

numeric_columns = final_feature_dataset.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_stats = (
    final_feature_dataset[numeric_columns]
    .describe()
    .T[["mean", "std", "min", "max"]]
    .rename(columns={
        "mean": "Mean",
        "std": "Std",
        "min": "Min",
        "max": "Max"
    })
)

feature_summary = feature_summary.merge(
    numeric_stats,
    left_on="Feature",
    right_index=True,
    how="left",
    validate="one_to_one"
)

print("ENHANCED FEATURE SUMMARY CREATED")
print("Total features:", len(feature_summary))
print("Total columns:", len(feature_summary.columns))

print("\nSummary columns:")
print(feature_summary.columns.tolist())

print("\nNumeric feature examples:")
display(
    feature_summary[
        feature_summary["Feature"].isin([
            "BRI_rf_mean_mean",
            "EI_Temp_mean",
            "MI_gs_mean",
            "PEI_HR_mean",
            "PRVI_HR_mean",
            "SHRI_hr_mean",
            "SII_mean",
            "SQI_SE_mean"
        ])
    ]
)

In [ ]:
# Visualize daily mobility measurements and historical rolling trend

plot_participant = "GER-001"

plot_df = final_feature_dataset.loc[
    final_feature_dataset["Participant_ID"] == plot_participant,
    [
        "Date",
        "MI_gs_mean",
        "MI_gs_mean_rolling_7d_mean"
    ]
].sort_values("Date").copy()

print("PARTICIPANT:", plot_participant)
print("Total dates:", len(plot_df))
print("Available daily measurements:", plot_df["MI_gs_mean"].notna().sum())
print("Available rolling means:", plot_df["MI_gs_mean_rolling_7d_mean"].notna().sum())

plt.figure(figsize=(14, 5))

plt.plot(
    plot_df["Date"],
    plot_df["MI_gs_mean"],
    marker="o",
    markersize=3,
    linewidth=1,
    label="Daily MI gs_mean"
)

plt.plot(
    plot_df["Date"],
    plot_df["MI_gs_mean_rolling_7d_mean"],
    linewidth=2,
    label="Previous 7-day rolling mean"
)

plt.title(f"Daily Mobility Measurement and Historical Trend - {plot_participant}")
plt.xlabel("Date")
plt.ylabel("MI gs_mean")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare final feature output directory

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("FEATURE OUTPUT DIRECTORY:")
print(FEATURES_DIR)

print("\nDirectory exists:", FEATURES_DIR.exists())
print("Is directory:", FEATURES_DIR.is_dir())

print("\nExisting files:")
for file in sorted(FEATURES_DIR.iterdir()):
    print(file.name)

In [ ]:
# Prepare daily sensor features for saving

daily_sensor_features_to_save = final_daily.drop(
    columns=["BRI_rf_mean_std"],
    errors="ignore"
).copy()

print("DAILY SENSOR FEATURES READY")
print("Rows:", len(daily_sensor_features_to_save))
print("Columns:", len(daily_sensor_features_to_save.columns))
print("Duplicate participant-date keys:",
      daily_sensor_features_to_save.duplicated(
          subset=["Participant_ID", "Date"]
      ).sum())

print("\nCompletely empty columns:",
      daily_sensor_features_to_save.columns[
          daily_sensor_features_to_save.isna().all()
      ].tolist())

In [ ]:
# Save daily sensor features

daily_output_path = FEATURES_DIR / "daily_sensor_features.parquet"

daily_sensor_features_to_save.to_parquet(
    daily_output_path,
    index=False
)

print("DAILY SENSOR FEATURES SAVED")
print("File:", daily_output_path)
print("File exists:", daily_output_path.exists())
print("File size (MB):", round(daily_output_path.stat().st_size / (1024 * 1024), 2))

In [ ]:
# Save rolling features

rolling_output_path = FEATURES_DIR / "rolling_features.parquet"

rolling_features.to_parquet(
    rolling_output_path,
    index=False
)

print("ROLLING FEATURES SAVED")
print("File:", rolling_output_path)
print("File exists:", rolling_output_path.exists())
print("File size (MB):", round(rolling_output_path.stat().st_size / (1024 * 1024), 2))
print("Rows:", len(rolling_features))
print("Columns:", len(rolling_features.columns))

In [ ]:
# Save historical baseline features

baseline_output_path = FEATURES_DIR / "baseline_features.parquet"

baseline_features.to_parquet(
    baseline_output_path,
    index=False
)

print("BASELINE FEATURES SAVED")
print("File:", baseline_output_path)
print("File exists:", baseline_output_path.exists())
print("File size (MB):", round(baseline_output_path.stat().st_size / (1024 * 1024), 2))
print("Rows:", len(baseline_features))
print("Columns:", len(baseline_features.columns))

In [ ]:
# Save data-quality features

quality_output_path = FEATURES_DIR / "data_quality_features.parquet"

data_quality_features.to_parquet(
    quality_output_path,
    index=False
)

print("DATA-QUALITY FEATURES SAVED")
print("File:", quality_output_path)
print("File exists:", quality_output_path.exists())
print("File size (MB):", round(quality_output_path.stat().st_size / (1024 * 1024), 2))
print("Rows:", len(data_quality_features))
print("Columns:", len(data_quality_features.columns))

In [ ]:
# Save cross-sensor features

cross_output_path = FEATURES_DIR / "cross_sensor_features.parquet"

cross_sensor_features.to_parquet(
    cross_output_path,
    index=False
)

print("CROSS-SENSOR FEATURES SAVED")
print("File:", cross_output_path)
print("File exists:", cross_output_path.exists())
print("File size (MB):", round(cross_output_path.stat().st_size / (1024 * 1024), 2))
print("Rows:", len(cross_sensor_features))
print("Columns:", len(cross_sensor_features.columns))

In [ ]:
# Save final integrated feature dataset

final_output_path = FEATURES_DIR / "final_feature_dataset.parquet"

final_feature_dataset.to_parquet(
    final_output_path,
    index=False
)

print("FINAL FEATURE DATASET SAVED")
print("File:", final_output_path)
print("File exists:", final_output_path.exists())
print("File size (MB):", round(final_output_path.stat().st_size / (1024 * 1024), 2))
print("Rows:", len(final_feature_dataset))
print("Columns:", len(final_feature_dataset.columns))
print("Duplicate participant-date keys:",
      final_feature_dataset.duplicated(
          subset=["Participant_ID", "Date"]
      ).sum())

In [ ]:
# Refresh and save feature summary

feature_summary["Data_Type"] = [
    str(final_feature_dataset[col].dtype)
    for col in feature_summary["Feature"]
]

summary_output_path = FEATURES_DIR / "feature_summary.csv"

feature_summary.to_csv(
    summary_output_path,
    index=False
)

print("FEATURE SUMMARY SAVED")
print("File:", summary_output_path)
print("File exists:", summary_output_path.exists())
print("Rows:", len(feature_summary))
print("Columns:", len(feature_summary.columns))
print("File size (KB):", round(summary_output_path.stat().st_size / 1024, 2))

In [ ]:
# Save feature dictionary

dictionary_output_path = FEATURES_DIR / "feature_dictionary.csv"

feature_dictionary.to_csv(
    dictionary_output_path,
    index=False
)

print("FEATURE DICTIONARY SAVED")
print("File:", dictionary_output_path)
print("File exists:", dictionary_output_path.exists())
print("Rows:", len(feature_dictionary))
print("Columns:", len(feature_dictionary.columns))
print("File size (KB):", round(dictionary_output_path.stat().st_size / 1024, 2))

In [ ]:
saved_dictionary = pd.read_csv(dictionary_output_path)

saved_pct = saved_dictionary[
    saved_dictionary["Feature"].str.endswith("_pct_change")
]

print("Saved percentage-change entries:", len(saved_pct))

print("\nSaved formula distribution:")
print(saved_pct["Formula"].value_counts())

print("\nSaved description distribution:")
print(saved_pct["Description"].value_counts())

In [ ]:
# Final saved-file validation

saved_files = {
    "daily_sensor_features.parquet": (13085, 152),
    "rolling_features.parquet": (13085, 578),
    "baseline_features.parquet": (13085, 194),
    "data_quality_features.parquet": (13085, 18),
    "cross_sensor_features.parquet": (13085, 4),
    "final_feature_dataset.parquet": (13085, 938),
    "feature_summary.csv": (938, 9),
    "feature_dictionary.csv": (938, 9)
}

validation_results = []

for filename, expected_shape in saved_files.items():
    file_path = FEATURES_DIR / filename

    if filename.endswith(".parquet"):
        loaded_df = pd.read_parquet(file_path)
    else:
        loaded_df = pd.read_csv(file_path)

    actual_shape = loaded_df.shape

    validation_results.append({
        "File": filename,
        "Expected_Shape": expected_shape,
        "Actual_Shape": actual_shape,
        "File_Exists": file_path.exists(),
        "Shape_Matches": actual_shape == expected_shape
    })

saved_files_validation = pd.DataFrame(validation_results)

display(saved_files_validation)

print("\nTOTAL FILES:", len(saved_files_validation))
print("FILES EXIST:", saved_files_validation["File_Exists"].all())
print("ALL SHAPES MATCH:", saved_files_validation["Shape_Matches"].all())
print(
    "VALIDATION STATUS:",
    "PASSED" if (
        saved_files_validation["File_Exists"].all()
        and saved_files_validation["Shape_Matches"].all()
    ) else "FAILED"
)

In [ ]:
# Inspect Git ignore rules

gitignore_path = PROJECT_ROOT / ".gitignore"

print("GITIGNORE PATH:", gitignore_path)
print("File exists:", gitignore_path.exists())

if gitignore_path.exists():
    print("\nCURRENT .gitignore CONTENT:\n")
    print(gitignore_path.read_text(encoding="utf-8"))
else:
    print("\nNo .gitignore file found.")

In [ ]:
# Verify feature outputs are ignored by Git

import subprocess

check_paths = [
    "data/processed/features/daily_sensor_features.parquet",
    "data/processed/features/rolling_features.parquet",
    "data/processed/features/baseline_features.parquet",
    "data/processed/features/data_quality_features.parquet",
    "data/processed/features/cross_sensor_features.parquet",
    "data/processed/features/final_feature_dataset.parquet",
    "data/processed/features/feature_summary.csv",
    "data/processed/features/feature_dictionary.csv"
]

print("GIT IGNORE VERIFICATION\n")

for path in check_paths:
    result = subprocess.run(
        ["git", "check-ignore", "-v", path],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    print(path)
    print("Ignored by Git:", result.returncode == 0)
    print()

In [ ]:
# Check Git working-tree status

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print("GIT WORKING-TREE STATUS\n")

if result.stdout.strip():
    print(result.stdout)
else:
    print("Working tree is clean.")

if result.stderr:
    print("\nGit errors:")
    print(result.stderr)

In [ ]:
# Inspect feature-engineering notebook

notebook_path = PROJECT_ROOT / "notebooks" / "04_feature_engineering.ipynb"

print("NOTEBOOK EXISTS:", notebook_path.exists())

if notebook_path.exists():
    print("Notebook size (KB):",
          round(notebook_path.stat().st_size / 1024, 2))

In [ ]:

# STEP 1: Inspect existing rolling feature pipeline

required_objects = [
    "final_daily",
    "rolling_features",
    "rolling_source_features",
    "rolling_feature_columns"
]

print("EXISTING OBJECT CHECK")
print("-" * 45)

for name in required_objects:
    exists = name in globals()
    print(f"{name}: {'Available' if exists else 'Missing'}")

print("\nROLLING FEATURE DATASET")
print("-" * 45)

if "rolling_features" in globals():
    print("Shape:", rolling_features.shape)
    print("Participants:", rolling_features["Participant_ID"].nunique())
    print("Duplicate keys:",
          rolling_features.duplicated(
              ["Participant_ID", "Date"]
          ).sum())

    print("\nExisting rolling feature examples:")
    rolling_columns = [
        col for col in rolling_features.columns
        if "_rolling_" in col
    ]
    print("\n".join(rolling_columns[:20]))
    print("\nTotal rolling feature columns:", len(rolling_columns))

print("\nSOURCE MEASUREMENT FEATURES")
print("-" * 45)

if "rolling_feature_columns" in globals():
    print("Total source features:", len(rolling_feature_columns))
    print(rolling_feature_columns[:15])


In [ ]:

# STEP 2: Analyze rolling-window observation coverage

print("ROLLING WINDOW COVERAGE ANALYSIS")
print("=" * 65)

for window in [3, 7, 14]:

    count_columns = [
        f"{feature}_rolling_{window}d_count"
        for feature in rolling_feature_columns
    ]

    count_data = rolling_features[count_columns]

    all_counts = count_data.to_numpy().flatten()
    all_counts = all_counts[~np.isnan(all_counts)]

    print(f"\n{window}-DAY WINDOW")
    print("-" * 45)

    print("Total feature-day observations:", len(all_counts))
    print("Zero-observation windows:", (all_counts == 0).sum())
    print("Windows with at least 1 observation:",
          (all_counts >= 1).sum())
    print("Windows with at least 2 observations:",
          (all_counts >= 2).sum())
    print("Windows with at least 3 observations:",
          (all_counts >= 3).sum())

    print("Median observation count:", np.median(all_counts))
    print("Maximum observation count:", np.max(all_counts))

    print("\nCoverage percentages:")

    for threshold in [1, 2, 3, 4, 5, 7]:
        if threshold <= window:
            percentage = (all_counts >= threshold).mean() * 100
            print(
                f"At least {threshold:2d} valid days: "
                f"{percentage:.2f}%"
            )


In [ ]:

# STEP 3: Optimized rolling feature rebuild

print("REBUILDING ROLLING FEATURES")
print("=" * 65)

window_thresholds = {
    3: {"mean": 1, "std": 2, "min": 1, "max": 1},
    7: {"mean": 2, "std": 2, "min": 2, "max": 2},
    14: {"mean": 3, "std": 3, "min": 3, "max": 3}
}

rolling_feature_frames = []

for participant_id, group in final_daily.groupby("Participant_ID"):

    group = group.sort_values("Date").copy()

    # Store all columns here; avoid repeated DataFrame insertion
    feature_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    date_indexed = group.set_index("Date")

    for feature in rolling_feature_columns:

        # Exclude current date to prevent data leakage
        historical_values = date_indexed[feature].shift(1)

        for window, thresholds in window_thresholds.items():

            rolling_window = historical_values.rolling(
                f"{window}D",
                min_periods=1
            )

            observation_count = rolling_window.count()

            prefix = f"{feature}_rolling_{window}d"

            feature_data[f"{prefix}_count"] = (
                observation_count.to_numpy()
            )

            for statistic in ["mean", "std", "min", "max"]:

                values = getattr(rolling_window, statistic)()

                values = values.where(
                    observation_count >= thresholds[statistic]
                )

                feature_data[f"{prefix}_{statistic}"] = (
                    values.to_numpy()
                )

    participant_features = pd.DataFrame(feature_data)
    rolling_feature_frames.append(participant_features)

rolling_features = pd.concat(
    rolling_feature_frames,
    ignore_index=True
)

print("Updated rolling feature shape:", rolling_features.shape)
print("Participants:", rolling_features["Participant_ID"].nunique())
print("Duplicate participant-date keys:",
      rolling_features.duplicated(["Participant_ID", "Date"]).sum())
print("Total rolling feature columns:", rolling_features.shape[1] - 2)

print("\nRolling feature examples:")
print(rolling_features.columns[2:18].tolist())


In [ ]:

# STEP 4: Add historical rolling change and percentage change

print("CREATING HISTORICAL CHANGE FEATURES")
print("=" * 65)

def calculate_last_change(values):
    valid = values[np.isfinite(values)]

    if len(valid) < 2:
        return np.nan

    return valid[-1] - valid[-2]


def calculate_last_pct_change(values):
    valid = values[np.isfinite(values)]

    if len(valid) < 2:
        return np.nan

    previous = valid[-2]
    latest = valid[-1]

    # Percentage change is undefined when the previous value is zero
    if previous == 0:
        return np.nan

    return ((latest - previous) / previous) * 100


change_frames = []

for participant_id, group in final_daily.groupby("Participant_ID"):

    group = group.sort_values("Date").copy()
    date_indexed = group.set_index("Date")

    feature_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    for feature in rolling_feature_columns:

        # Exclude current date: only past observations are considered
        historical_values = date_indexed[feature].shift(1)

        for window in [3, 7, 14]:

            rolling_window = historical_values.rolling(
                f"{window}D",
                min_periods=1
            )

            prefix = f"{feature}_rolling_{window}d"

            feature_data[f"{prefix}_change"] = (
                rolling_window.apply(
                    calculate_last_change,
                    raw=True
                ).to_numpy()
            )

            feature_data[f"{prefix}_pct_change"] = (
                rolling_window.apply(
                    calculate_last_pct_change,
                    raw=True
                ).to_numpy()
            )

    change_frames.append(pd.DataFrame(feature_data))

change_features = pd.concat(change_frames, ignore_index=True)

# Remove previous versions if this cell is rerun
new_columns = [
    column for column in change_features.columns
    if column not in ["Participant_ID", "Date"]
]

rolling_features = rolling_features.drop(
    columns=[
        column for column in new_columns
        if column in rolling_features.columns
    ],
    errors="ignore"
)

# Merge using participant and date keys, never row positions
rolling_features = rolling_features.merge(
    change_features,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one",
    sort=False
)

print("Change features created:", len(new_columns))
print("Updated rolling feature shape:", rolling_features.shape)
print("Duplicate participant-date keys:",
      rolling_features.duplicated(["Participant_ID", "Date"]).sum())

print("\nExample change features:")
print([
    column for column in rolling_features.columns
    if column.endswith("_change") or column.endswith("_pct_change")
][:12])


In [ ]:

# STEP 5: Validate rolling coverage, changes, and historical-only calculations

print("ROLLING FEATURE VALIDATION")
print("=" * 65)

# 1. Check minimum coverage rules
coverage_violations = 0
change_violations = 0

for feature in rolling_feature_columns:
    for window, thresholds in window_thresholds.items():

        prefix = f"{feature}_rolling_{window}d"
        count = rolling_features[f"{prefix}_count"]

        for statistic in ["mean", "std", "min", "max"]:
            values = rolling_features[f"{prefix}_{statistic}"]
            invalid = values.notna() & (count < thresholds[statistic])
            coverage_violations += int(invalid.sum())

        for statistic in ["change", "pct_change"]:
            values = rolling_features[f"{prefix}_{statistic}"]
            invalid = values.notna() & (count < 2)
            change_violations += int(invalid.sum())

print("Minimum coverage violations:", coverage_violations)
print("Change coverage violations:", change_violations)

# 2. Check infinities
numeric_data = rolling_features.select_dtypes(include=np.number)
infinite_values = np.isinf(numeric_data.to_numpy()).sum()

print("Infinite numeric values:", infinite_values)

# 3. Verify one change calculation against raw historical values
test_feature = rolling_feature_columns[0]
test_window = 7
test_prefix = f"{test_feature}_rolling_{test_window}d"

test_rows = rolling_features[
    (rolling_features[f"{test_prefix}_count"] >= 2)
    & rolling_features[f"{test_prefix}_change"].notna()
]

if len(test_rows) > 0:

    test_row = test_rows.iloc[0]
    test_id = test_row["Participant_ID"]
    test_date = pd.Timestamp(test_row["Date"])

    source_group = final_daily[
        final_daily["Participant_ID"] == test_id
    ].copy()

    source_group["Date"] = pd.to_datetime(source_group["Date"])

    historical = source_group[
        (source_group["Date"] < test_date)
        & (source_group["Date"] > test_date - pd.Timedelta(days=test_window))
    ][test_feature].dropna()

    expected_change = historical.iloc[-1] - historical.iloc[-2]

    actual_change = test_row[f"{test_prefix}_change"]

    print("\nCHANGE CALCULATION SPOT CHECK")
    print("Participant:", test_id)
    print("Feature:", test_feature)
    print("Date:", test_date)
    print("Historical observations:", len(historical))
    print("Expected change:", expected_change)
    print("Calculated change:", actual_change)
    print("Match:", np.isclose(expected_change, actual_change))

else:
    print("\nNo suitable row found for the change spot check.")

# 4. Check duplicate keys
print("\nDuplicate participant-date keys:",
      rolling_features.duplicated(["Participant_ID", "Date"]).sum())


In [ ]:

print("EXISTING BASELINE FEATURE STRUCTURE")
print("=" * 60)

print("Shape:", baseline_features.shape)

print("\nExample columns:")
print(baseline_features.columns[:25].tolist())

print("\nBaseline feature examples:")
for column in baseline_features.columns:
    if any(term in column for term in ["historical_mean", "historical_std", "historical_count"]):
        print(column)
        if column.endswith("historical_count"):
            break


In [ ]:

# STEP 7: Add historical median, deviation and standardized deviation

print("BUILDING ADDITIONAL BASELINE FEATURES")
print("=" * 65)

baseline_extra_frames = []

for participant_id, group in final_daily.groupby("Participant_ID"):

    group = group.sort_values("Date").copy()
    date_indexed = group.set_index("Date")

    feature_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    for feature in rolling_feature_columns:

        historical = date_indexed[feature].shift(1)

        historical_count = historical.expanding(
            min_periods=1
        ).count()

        historical_mean = historical.expanding(
            min_periods=3
        ).mean()

        historical_median = historical.expanding(
            min_periods=3
        ).median()

        historical_std = historical.expanding(
            min_periods=3
        ).std()

        current_values = date_indexed[feature]

        enough_history = historical_count >= 3

        prefix = feature

        feature_data[f"{prefix}_baseline_median"] = (
            historical_median.to_numpy()
        )

        deviation = (current_values - historical_mean).where(
            enough_history
        )

        feature_data[f"{prefix}_baseline_deviation"] = (
            deviation.to_numpy()
        )

        standardized_deviation = (
            deviation / historical_std.replace(0, np.nan)
        )

        feature_data[
            f"{prefix}_baseline_standardized_deviation"
        ] = standardized_deviation.to_numpy()

    baseline_extra_frames.append(pd.DataFrame(feature_data))

baseline_extra_features = pd.concat(
    baseline_extra_frames,
    ignore_index=True
)

extra_columns = [
    column for column in baseline_extra_features.columns
    if column not in ["Participant_ID", "Date"]
]

baseline_features = baseline_features.merge(
    baseline_extra_features,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

print("Additional baseline features:", len(extra_columns))
print("Updated baseline shape:", baseline_features.shape)
print("Duplicate participant-date keys:",
      baseline_features.duplicated(["Participant_ID", "Date"]).sum())


In [ ]:

# FIX: Baseline calculation spot check

test_feature = rolling_feature_columns[0]

test_row = baseline_features[
    baseline_features[f"{test_feature}_baseline_count"] >= 3
].iloc[0]

test_id = test_row["Participant_ID"]
test_date = pd.Timestamp(test_row["Date"])

source_group = final_daily[
    final_daily["Participant_ID"] == test_id
].copy()

source_group["Date"] = pd.to_datetime(source_group["Date"])

# Get only valid historical observations before the test date
historical = source_group.loc[
    source_group["Date"] < test_date,
    ["Date", test_feature]
].dropna(subset=[test_feature]).sort_values("Date")

expected_median = historical[test_feature].median()
expected_mean = historical[test_feature].mean()

# Get current day's measurement from final_daily
current_value = source_group.loc[
    source_group["Date"] == test_date,
    test_feature
].iloc[0]

expected_deviation = current_value - expected_mean

print("BASELINE SPOT CHECK")
print("=" * 60)
print("Participant:", test_id)
print("Feature:", test_feature)
print("Date:", test_date)
print("Historical observations:", len(historical))

print("Median matches:",
      np.isclose(
          expected_median,
          test_row[f"{test_feature}_baseline_median"]
      ))

print("Deviation matches:",
      np.isclose(
          expected_deviation,
          test_row[f"{test_feature}_baseline_deviation"]
      ))

print("Duplicate keys:",
      baseline_features.duplicated(
          ["Participant_ID", "Date"]
      ).sum())


In [ ]:

# DIAGNOSE BASELINE DEVIATION MISMATCH

print("BASELINE DEVIATION DIAGNOSTIC")
print("=" * 60)

print("Current measurement:", current_value)
print("Historical mean:", expected_mean)
print("Expected deviation:", expected_deviation)

actual_deviation = test_row[
    f"{test_feature}_baseline_deviation"
]

print("Stored baseline deviation:", actual_deviation)
print("Historical observation count:", len(historical))

print("\nMissing-value checks:")
print("Current measurement missing:", pd.isna(current_value))
print("Expected deviation missing:", pd.isna(expected_deviation))
print("Stored deviation missing:", pd.isna(actual_deviation))

if pd.notna(expected_deviation) and pd.notna(actual_deviation):
    print("\nNumerical difference:",
          abs(expected_deviation - actual_deviation))


In [ ]:
print("EXISTING DATA-QUALITY FEATURES")
print("=" * 60)

print("Shape:", data_quality_features.shape)

print("\nColumns:")
print(data_quality_features.columns.tolist())

print("\nSample:")
print(data_quality_features.head(3).to_string(index=False))

In [ ]:

# STEP 10: Add 7-day observation counts and coverage percentages

print("ADDING 7-DAY DATA-QUALITY FEATURES")
print("=" * 65)

quality_frames = []

sensors = ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]

for participant_id, group in data_quality_features.groupby("Participant_ID"):

    group = group.sort_values("Date").copy()
    date_indexed = group.set_index("Date")

    feature_data = {
        "Participant_ID": group["Participant_ID"].to_numpy(),
        "Date": group["Date"].to_numpy()
    }

    for sensor in sensors:

        availability = date_indexed[f"{sensor}_availability"]

        # Only use the previous 7 calendar days
        historical_availability = availability.shift(1)

        rolling_count = historical_availability.rolling(
            "7D",
            min_periods=1
        ).sum()

        feature_data[
            f"{sensor}_valid_observation_count_7d"
        ] = rolling_count.to_numpy()

        feature_data[
            f"{sensor}_coverage_percentage_7d"
        ] = (rolling_count / 7 * 100).to_numpy()

    quality_frames.append(pd.DataFrame(feature_data))

quality_extra = pd.concat(quality_frames, ignore_index=True)

extra_columns = [
    column for column in quality_extra.columns
    if column not in ["Participant_ID", "Date"]
]

data_quality_features = data_quality_features.merge(
    quality_extra,
    on=["Participant_ID", "Date"],
    how="left",
    validate="one_to_one"
)

print("Additional features created:", len(extra_columns))
print("Updated data-quality shape:", data_quality_features.shape)

print("Duplicate participant-date keys:",
      data_quality_features.duplicated(
          ["Participant_ID", "Date"]
      ).sum())

print("\nNew feature examples:")
print(extra_columns[:8])


In [ ]:

# FIX: Find an available participant-sensor combination automatically

print("7-DAY DATA-QUALITY SPOT CHECK")
print("=" * 60)

test_row = None
test_sensor = None

for sensor in sensors:
    count_col = f"{sensor}_valid_observation_count_7d"

    candidates = data_quality_features[
        data_quality_features[count_col] > 0
    ]

    if not candidates.empty:
        test_row = candidates.iloc[0]
        test_sensor = sensor
        break

if test_row is not None:

    test_id = test_row["Participant_ID"]
    test_date = pd.Timestamp(test_row["Date"])

    test_group = data_quality_features[
        data_quality_features["Participant_ID"] == test_id
    ].copy()

    historical_rows = test_group[
        (test_group["Date"] < test_date)
        & (test_group["Date"] > test_date - pd.Timedelta(days=7))
    ]

    expected_count = historical_rows[
        f"{test_sensor}_availability"
    ].sum()

    actual_count = test_row[
        f"{test_sensor}_valid_observation_count_7d"
    ]

    expected_percentage = expected_count / 7 * 100

    actual_percentage = test_row[
        f"{test_sensor}_coverage_percentage_7d"
    ]

    print("Participant:", test_id)
    print("Sensor:", test_sensor)
    print("Date:", test_date)
    print("Expected count:", expected_count)
    print("Calculated count:", actual_count)
    print("Count matches:", expected_count == actual_count)
    print("Expected coverage:", expected_percentage)
    print("Calculated coverage:", actual_percentage)
    print("Coverage matches:",
          np.isclose(expected_percentage, actual_percentage))

else:
    print("No positive 7-day observation counts found.")

print("\nDuplicate keys:",
      data_quality_features.duplicated(
          ["Participant_ID", "Date"]
      ).sum())


In [ ]:
print("CROSS-SENSOR VARIABLE CHECK")
print("=" * 60)

for name in [
    "cross_sensor_features",
    "cross_sensor",
    "final_feature_dataset",
    "final_daily"
]:
    print(name, ":", name in globals())

print("\nExisting cross-sensor columns in final dataset:")

if "final_feature_dataset" in globals():
    print([
        col for col in final_feature_dataset.columns
        if "both_available" in col or "difference" in col
    ])

In [ ]:
# STEP 12B: Check cross-sensor overlap

candidate_pairs = [
    ("BRI_rf_mean_mean", "PEI_SpO2_mean"),
    ("MI_gs_mean", "SQI_sleep_duration_mean"),
    ("EI_AQI_mean", "BRI_rf_mean_mean"),
    ("EI_AQI_mean", "PEI_HR_mean"),
    ("PEI_HR_mean", "PRVI_HR_mean")
]

print("CROSS-SENSOR OVERLAP ANALYSIS")
print("=" * 65)

for feature_a, feature_b in candidate_pairs:

    if feature_a in final_daily.columns and feature_b in final_daily.columns:

        valid = (
            final_daily[feature_a].notna()
            & final_daily[feature_b].notna()
        )

        overlap = final_daily.loc[valid]

        print(f"\n{feature_a} + {feature_b}")
        print("Overlapping participant-days:", len(overlap))
        print("Unique participants:",
              overlap["Participant_ID"].nunique())

    else:
        print(f"\n{feature_a} + {feature_b}: Column unavailable")

In [ ]:
# STEP 12C: Create cross-sensor features

import pandas as pd
from pathlib import Path

cross_sensor_features = final_daily[
    ["Participant_ID", "Date"]
].copy()

cross_sensor_pairs = {
    "BRI_PEI_SpO2": ("BRI_rf_mean_mean", "PEI_SpO2_mean"),
    "MI_SQI_Sleep": ("MI_gs_mean", "SQI_sleep_duration_mean"),
    "EI_BRI_AQI": ("EI_AQI_mean", "BRI_rf_mean_mean"),
    "EI_PEI_HR": ("EI_AQI_mean", "PEI_HR_mean"),
    "PEI_PRVI_HR": ("PEI_HR_mean", "PRVI_HR_mean")
}

for pair_name, (feature_a, feature_b) in cross_sensor_pairs.items():

    both_available = (
        final_daily[feature_a].notna()
        & final_daily[feature_b].notna()
    )

    cross_sensor_features[f"{pair_name}_both_available"] = (
        both_available.astype("int8")
    )

    # Difference only for comparable heart-rate measurements
    if pair_name == "PEI_PRVI_HR":
        cross_sensor_features[
            "PEI_PRVI_HR_difference"
        ] = (
            final_daily[feature_a] - final_daily[feature_b]
        )

# Save output
feature_dir = Path("data/processed/features")
feature_dir.mkdir(parents=True, exist_ok=True)

cross_sensor_features.to_parquet(
    feature_dir / "cross_sensor_features.parquet",
    index=False
)

print("CROSS-SENSOR FEATURES CREATED")
print("Shape:", cross_sensor_features.shape)
print("Columns:", cross_sensor_features.columns.tolist())
print("Duplicate keys:",
      cross_sensor_features.duplicated(
          ["Participant_ID", "Date"]
      ).sum())
print("\nAvailability counts:")

for column in cross_sensor_features.columns:
    if column.endswith("_both_available"):
        print(column, ":", cross_sensor_features[column].sum())

print("\nSaved:", feature_dir / "cross_sensor_features.parquet")

In [ ]:
# STEP 13: Build the integrated feature dataset

key_columns = ["Participant_ID", "Date"]

feature_groups = {
    "daily": final_daily,
    "rolling": rolling_features,
    "baseline": baseline_features,
    "data_quality": data_quality_features,
    "cross_sensor": cross_sensor_features
}

print("FEATURE GROUP VALIDATION")
print("=" * 60)

for name, df in feature_groups.items():
    print(
        f"{name}: Shape={df.shape}, "
        f"Duplicate keys={df.duplicated(key_columns).sum()}"
    )

    assert set(key_columns).issubset(df.columns), (
        f"Missing key columns in {name}"
    )

    assert not df.duplicated(key_columns).any(), (
        f"Duplicate participant-date keys in {name}"
    )

# Check for overlapping feature names
all_feature_columns = []

for df in feature_groups.values():
    all_feature_columns.extend(
        col for col in df.columns if col not in key_columns
    )

duplicate_columns = sorted({
    col for col in all_feature_columns
    if all_feature_columns.count(col) > 1
})

print("\nOverlapping feature columns:", duplicate_columns)

assert not duplicate_columns, "Resolve overlapping feature names first."

# Merge all feature groups using participant and date keys
final_feature_dataset = final_daily.copy()

for name in ["rolling", "baseline", "data_quality", "cross_sensor"]:
    final_feature_dataset = final_feature_dataset.merge(
        feature_groups[name],
        on=key_columns,
        how="left",
        validate="one_to_one"
    )

print("\nFINAL INTEGRATED FEATURE DATASET")
print("=" * 60)
print("Shape:", final_feature_dataset.shape)
print("Duplicate keys:",
      final_feature_dataset.duplicated(key_columns).sum())
print("Unique participants:",
      final_feature_dataset["Participant_ID"].nunique())
print("Date range:",
      final_feature_dataset["Date"].min(),
      "to",
      final_feature_dataset["Date"].max())
print("Missing feature columns:",
      final_feature_dataset.iloc[:, 2:].isna().all().sum())

In [ ]:
# STEP 13B: Inspect completely missing columns

empty_columns = [
    col for col in final_feature_dataset.columns
    if final_feature_dataset[col].isna().all()
]

print("COMPLETELY EMPTY COLUMNS")
print("=" * 60)
print("Count:", len(empty_columns))
print("Columns:", empty_columns)

print("\nDATASET VALIDATION")
print("=" * 60)
print("Total rows:", len(final_feature_dataset))
print("Total columns:", len(final_feature_dataset.columns))
print("Duplicate keys:",
      final_feature_dataset.duplicated(
          ["Participant_ID", "Date"]
      ).sum())
print("Infinite numeric values:",
      np.isinf(
          final_feature_dataset.select_dtypes(include="number")
          .to_numpy()
      ).sum())

In [ ]:
# STEP 13C: Remove the fully empty feature

final_feature_dataset = final_feature_dataset.drop(
    columns=["BRI_rf_mean_std"]
)

print("FINAL DATASET AFTER CLEANUP")
print("=" * 60)
print("Shape:", final_feature_dataset.shape)
print("Duplicate keys:",
      final_feature_dataset.duplicated(
          ["Participant_ID", "Date"]
      ).sum())
print("Completely empty columns:",
      final_feature_dataset.columns[
          final_feature_dataset.isna().all()
      ].tolist())
print("Infinite numeric values:",
      np.isinf(
          final_feature_dataset.select_dtypes(include="number")
          .to_numpy()
      ).sum())

In [ ]:
# STEP 14: Inspect existing metadata

print("METADATA VARIABLE CHECK")
print("=" * 60)

for name in ["feature_summary", "feature_dictionary"]:
    print(f"{name} available:", name in globals())

if "feature_dictionary" in globals():
    print("\nExisting dictionary shape:", feature_dictionary.shape)
    print("Existing dictionary columns:")
    print(feature_dictionary.columns.tolist())

if "feature_summary" in globals():
    print("\nExisting summary shape:", feature_summary.shape)
    print("Existing summary columns:")
    print(feature_summary.columns.tolist())

print("\nFinal dataset feature count:",
      len(final_feature_dataset.columns) - 2)

In [ ]:
# STEP 14A: Generate feature summary

summary_records = []

for col in final_feature_dataset.columns:

    if col in ["Participant_ID", "Date"]:
        continue

    series = final_feature_dataset[col]
    numeric = pd.to_numeric(series, errors="coerce")

    is_numeric = pd.api.types.is_numeric_dtype(series)

    record = {
        "Feature": col,
        "Data_Type": str(series.dtype),
        "Missing_Count": int(series.isna().sum()),
        "Missing_Percentage": round(
            series.isna().mean() * 100, 2
        ),
        "Unique_Values": int(series.nunique()),
        "Mean": numeric.mean() if is_numeric else np.nan,
        "Std": numeric.std() if is_numeric else np.nan,
        "Min": numeric.min() if is_numeric else np.nan,
        "Max": numeric.max() if is_numeric else np.nan
    }

    summary_records.append(record)

feature_summary = pd.DataFrame(summary_records)

print("FEATURE SUMMARY CREATED")
print("=" * 60)
print("Shape:", feature_summary.shape)
print("Expected features:", 1916)
print("Actual features:", len(feature_summary))
print("\nMissingness overview:")
print(
    feature_summary["Missing_Percentage"]
    .describe()
    .round(2)
)
print("\nFirst 10 records:")
display(feature_summary.head(10))

In [ ]:
# STEP 14B: Inspect feature names by group

groups_to_inspect = {
    "Daily": final_daily,
    "Rolling": rolling_features,
    "Baseline": baseline_features,
    "Data Quality": data_quality_features,
    "Cross Sensor": cross_sensor_features
}

for group_name, df in groups_to_inspect.items():

    features = [
        col for col in df.columns
        if col not in ["Participant_ID", "Date"]
    ]

    print(f"\n{group_name.upper()} FEATURES")
    print("-" * 60)
    print("Total:", len(features))

    for feature in features[:12]:
        print(feature)

In [ ]:

# STEP 14C: Generate feature dictionary

import re

sensors = ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]

dictionary_records = []

for feature in final_feature_dataset.columns:

    if feature in ["Participant_ID", "Date"]:
        continue

    # Identify feature group and source sensor
    if feature.endswith("_difference") or "_both_available" in feature:
        group = "Cross-Sensor"

        if feature == "PEI_PRVI_HR_difference":
            source = "PEI, PRVI"
            description = "Difference between PEI and PRVI heart-rate measurements."
            formula = "PEI_HR_mean - PRVI_HR_mean"
            units = "As defined in source datasets; confirmation required"
        else:
            source = ", ".join(
                s for s in sensors if s in feature.split("_")
            )
            description = (
                "Indicator showing whether both corresponding "
                "sensor measurements are available."
            )
            formula = "1 if both measurements are available; otherwise 0"
            units = "Binary indicator"

        window = "Same participant-date"

        missing_handling = (
            "Missing source measurements remain missing; "
            "availability indicator is 1 only when both exist."
        )

        leakage = (
            "Uses measurements from the same date; suitable for "
            "same-day analysis, not automatically available before "
            "those measurements are collected."
        )

    elif "_rolling_" in feature:
        group = "Rolling"

        source = next(
            (s for s in sensors if feature.startswith(s + "_")),
            "Unknown"
        )

        match = re.search(r"_rolling_(3|7|14)d_", feature)
        window_days = int(match.group(1)) if match else None
        window = (
            f"Previous {window_days} calendar days"
            if window_days else "See feature name"
        )

        # Check pct_change before change because pct_change
        # also ends with the word change.
        if feature.endswith("_pct_change"):
            statistic = "pct_change"
        else:
            statistic = feature.rsplit("_", 1)[-1]

        descriptions = {
            "count": "Number of valid historical observations in the rolling window.",
            "mean": "Mean of valid historical observations in the rolling window.",
            "std": "Standard deviation of valid historical observations in the rolling window.",
            "min": "Minimum valid historical observation in the rolling window.",
            "max": "Maximum valid historical observation in the rolling window.",
            "change": "Difference between the latest and second-latest valid historical observations.",
            "pct_change": "Percentage change between the latest and second-latest valid historical observations."
        }

        description = descriptions.get(
            statistic, "Historical rolling-window feature."
        )

        formula = {
            "count": "Count of valid observations",
            "mean": "Mean of valid observations",
            "std": "Standard deviation of valid observations",
            "min": "Minimum of valid observations",
            "max": "Maximum of valid observations",
            "change": "Latest historical value - previous historical value",
            "pct_change": "((Latest - Previous) / Previous) * 100; NaN when Previous is zero"
        }.get(statistic, "See feature name")

        units = (
            "Observation count" if statistic == "count"
            else "Percentage" if statistic == "pct_change"
            else "As defined in source dataset; confirmation required"
        )

        missing_handling = (
            "Missing observations excluded; minimum observation "
            "threshold applied for the corresponding window. "
            "Insufficient history remains NaN."
        )

        leakage = (
            "Historical observations only; current date excluded "
            "from rolling calculations."
        )

    elif "_baseline_" in feature:
        group = "Baseline"

        source = next(
            (s for s in sensors if feature.startswith(s + "_")),
            "Unknown"
        )

        statistic = feature.rsplit("_", 1)[-1]

        descriptions = {
            "mean": "Participant-specific expanding historical mean.",
            "std": "Participant-specific expanding historical standard deviation.",
            "count": "Number of valid historical observations.",
            "median": "Participant-specific expanding historical median.",
            "deviation": "Current measurement minus historical mean."
        }

        if feature.endswith("_baseline_standardized_deviation"):
            statistic = "standardized_deviation"

        descriptions["standardized_deviation"] = (
            "Current measurement's deviation from historical mean, "
            "divided by historical standard deviation."
        )

        description = descriptions.get(
            statistic, "Participant-specific historical baseline feature."
        )

        formula = {
            "mean": "Mean of prior valid observations",
            "std": "Standard deviation of prior valid observations",
            "count": "Count of prior valid observations",
            "median": "Median of prior valid observations",
            "deviation": "Current value - historical mean",
            "standardized_deviation": (
                "(Current value - historical mean) / historical standard deviation"
            )
        }.get(statistic, "See feature name")

        units = (
            "Observation count" if statistic == "count"
            else "Dimensionless standardized value"
            if statistic == "standardized_deviation"
            else "As defined in source dataset; confirmation required"
        )

        window = "Expanding participant-specific history"

        missing_handling = (
            "Historical statistics require at least 3 prior valid "
            "observations. Missing current measurements remain NaN."
        )

        leakage = (
            "Historical baseline uses prior observations only; "
            "current observation excluded from baseline calculation."
        )

    elif any(
        token in feature for token in [
            "_availability",
            "_days_since_last_valid_observation",
            "_valid_observation_count_7d",
            "_coverage_percentage_7d"
        ]
    ):
        group = "Data Quality"

        source = next(
            (s for s in sensors if feature.startswith(s + "_")),
            "Unknown"
        )

        if feature.endswith("_availability"):
            description = "Indicates whether a valid sensor observation is available."
            formula = "1 if valid observation is available; otherwise 0"
            units = "Binary indicator"
            window = "Current date"

        elif feature.endswith("_days_since_last_valid_observation"):
            description = "Calendar days since the latest valid sensor observation."
            formula = "Current date - date of latest valid observation"
            units = "Days"
            window = "Historical through current date"

        elif feature.endswith("_valid_observation_count_7d"):
            description = "Valid observation days in the preceding 7-day window."
            formula = "Sum of valid availability indicators over prior 7 days"
            units = "Observation days"
            window = "Previous 7 calendar days"

        else:
            description = "Percentage of days with valid sensor observations in the preceding 7-day window."
            formula = "(Valid observation days / 7) * 100"
            units = "Percentage"
            window = "Previous 7 calendar days"

        missing_handling = (
            "Availability and coverage derived from valid observation "
            "indicators; unavailable measurements are not replaced with zero."
        )

        leakage = (
            "Historical windows exclude the current date; current-date "
            "availability describes same-day data status."
        )

    else:
        group = "Daily"

        source = next(
            (s for s in sensors if feature.startswith(s + "_")),
            "Unknown"
        )

        if "observation_count" in feature:
            description = "Number of sensor observations recorded for the participant and date."
            formula = "Count of source observations"
            units = "Observations"

        elif "validity" in feature:
            description = "Source-provided sensor validity indicator."
            formula = "As defined by the source dataset"
            units = "Source-defined validity indicator"

        elif "_mean_" in feature:
            description = "Daily mean of the corresponding source measurement."
            formula = "Mean of available daily source measurements"
            units = "As defined in source dataset; confirmation required"

        elif "_std_" in feature:
            description = "Daily standard deviation of the corresponding source measurement."
            formula = "Standard deviation of available daily source measurements"
            units = "As defined in source dataset; confirmation required"

        elif "_min_" in feature:
            description = "Daily minimum of the corresponding source measurement."
            formula = "Minimum of available daily source measurements"
            units = "As defined in source dataset; confirmation required"

        elif "_max_" in feature:
            description = "Daily maximum of the corresponding source measurement."
            formula = "Maximum of available daily source measurements"
            units = "As defined in source dataset; confirmation required"

        else:
            description = "Daily sensor feature derived from the corresponding source dataset."
            formula = "As defined by daily feature engineering"
            units = "As defined in source dataset; confirmation required"

        window = "Current calendar day"

        missing_handling = (
            "Missing measurements remain NaN; no blind forward filling "
            "or replacement with zero."
        )

        leakage = (
            "Same-day feature; availability depends on the timing "
            "of source measurements."
        )

    dictionary_records.append({
        "Feature": feature,
        "Source_Sensor": source,
        "Feature_Group": group,
        "Description": description,
        "Formula": formula,
        "Units": units,
        "Time_Window": window,
        "Missing_Handling": missing_handling,
        "Leakage_Considerations": leakage
    })

feature_dictionary = pd.DataFrame(dictionary_records)

print("FEATURE DICTIONARY CREATED")
print("=" * 60)
print("Shape:", feature_dictionary.shape)
print("Expected features:", len(final_feature_dataset.columns) - 2)
print("Actual features:", len(feature_dictionary))

print("\nFeature group distribution:")
print(feature_dictionary["Feature_Group"].value_counts())

print("\nMissing dictionary values:")
print(feature_dictionary.isna().sum())

print("\nSample records:")
display(feature_dictionary.head(10))


In [ ]:
pct_change_dictionary = feature_dictionary[
    feature_dictionary["Feature"].str.endswith("_pct_change")
]

print("Percentage-change entries:", len(pct_change_dictionary))

print("\nFormula distribution:")
print(pct_change_dictionary["Formula"].value_counts())

print("\nDescription distribution:")
print(pct_change_dictionary["Description"].value_counts())

print("\nSample entries:")
display(
    pct_change_dictionary[
        ["Feature", "Description", "Formula"]
    ].head(10)
)

In [ ]:
# STEP 14D: Check dictionary descriptions and formulas

print("INVALID OBSERVATION FEATURES")
print("=" * 60)

display(
    feature_dictionary[
        feature_dictionary["Feature"].str.contains(
            "invalid_observation_count", case=False
        )
    ][["Feature", "Description", "Formula", "Units"]]
)

print("\nBASELINE FEATURE TYPES")
print("=" * 60)

baseline_names = feature_dictionary.loc[
    feature_dictionary["Feature_Group"] == "Baseline",
    "Feature"
]

for suffix in ["mean", "std", "count", "median",
               "deviation", "standardized_deviation"]:
    print(
        suffix,
        ":",
        baseline_names.str.endswith(
            "_baseline_" + suffix
        ).sum()
    )

print("\nCROSS-SENSOR FORMULAS")
print("=" * 60)

display(
    feature_dictionary[
        feature_dictionary["Feature_Group"] == "Cross-Sensor"
    ][["Feature", "Formula", "Units"]]
)

In [ ]:
# STEP 14E: Correct invalid-observation feature descriptions

invalid_mask = feature_dictionary["Feature"].str.endswith(
    "_invalid_observation_count"
)

feature_dictionary.loc[invalid_mask, "Description"] = (
    "Number of source observations marked invalid by the sensor "
    "validity indicators for the participant and date."
)

feature_dictionary.loc[invalid_mask, "Formula"] = (
    "Count of source observations classified as invalid"
)

feature_dictionary.loc[invalid_mask, "Units"] = "Observations"

feature_dictionary.loc[invalid_mask, "Missing_Handling"] = (
    "Invalid observations are counted separately from missing "
    "measurements; original validity information is preserved."
)

feature_dictionary.loc[invalid_mask, "Leakage_Considerations"] = (
    "Same-day data-quality feature; depends on source observations "
    "and their validity flags for that date."
)

print("CORRECTED INVALID-OBSERVATION FEATURES")
print("=" * 60)
print("Corrected entries:", invalid_mask.sum())

display(
    feature_dictionary.loc[
        invalid_mask,
        ["Feature", "Description", "Formula", "Units"]
    ]
)

print("\nDictionary shape:", feature_dictionary.shape)
print("Missing values:", feature_dictionary.isna().sum().sum())
print("Duplicate feature names:",
      feature_dictionary["Feature"].duplicated().sum())

In [ ]:
# STEP 15: Save all feature-engineering outputs

from pathlib import Path

output_dir = Path("data/processed/features")
output_dir.mkdir(parents=True, exist_ok=True)

# Remove the known fully empty feature from daily output
daily_output = final_daily.drop(
    columns=["BRI_rf_mean_std"],
    errors="ignore"
)

# Prepare all required outputs
outputs = {
    "daily_sensor_features.parquet": daily_output,
    "rolling_features.parquet": rolling_features,
    "baseline_features.parquet": baseline_features,
    "data_quality_features.parquet": data_quality_features,
    "cross_sensor_features.parquet": cross_sensor_features,
    "final_feature_dataset.parquet": final_feature_dataset,
    "feature_summary.csv": feature_summary,
    "feature_dictionary.csv": feature_dictionary
}

# Save outputs
for filename, df in outputs.items():

    filepath = output_dir / filename

    if filename.endswith(".parquet"):
        df.to_parquet(filepath, index=False)
    else:
        df.to_csv(filepath, index=False)

    print(f"Saved: {filepath} | Shape: {df.shape}")

print("\nALL FEATURE OUTPUTS SAVED")
print("=" * 60)
print("Output directory:", output_dir.resolve())
print("Total files:", len(outputs))

In [ ]:
# step15c corrected 
# Save outputs to the correct project directory

correct_output_dir = FEATURES_DIR
correct_output_dir.mkdir(parents=True, exist_ok=True)

for filename, df in outputs.items():

    filepath = correct_output_dir / filename

    if filename.endswith(".parquet"):
        df.to_parquet(filepath, index=False)
    else:
        df.to_csv(filepath, index=False)

    print(f"Saved: {filepath}")

print("\nOUTPUT DIRECTORY:")
print(correct_output_dir.resolve())


In [ ]:
# STEP 16: Read-back validation of all saved outputs

print("SAVED OUTPUT READ-BACK VALIDATION")
print("=" * 65)

validation_results = []

for filename, original_df in outputs.items():

    filepath = correct_output_dir / filename

    if filename.endswith(".parquet"):
        loaded_df = pd.read_parquet(filepath)
    else:
        loaded_df = pd.read_csv(filepath)

    shape_match = loaded_df.shape == original_df.shape

    validation_results.append({
        "File": filename,
        "Expected_Shape": original_df.shape,
        "Loaded_Shape": loaded_df.shape,
        "Shape_Match": shape_match
    })

    print(f"\n{filename}")
    print("Expected:", original_df.shape)
    print("Loaded:", loaded_df.shape)
    print("Match:", shape_match)

validation_report = pd.DataFrame(validation_results)

print("\nFINAL READ-BACK RESULT")
print("=" * 65)
print("Files checked:", len(validation_report))
print("Successful:", validation_report["Shape_Match"].sum())
print("Failed:", (~validation_report["Shape_Match"]).sum())

assert validation_report["Shape_Match"].all(), (
    "At least one saved file has a shape mismatch."
)

print("\nAll 8 saved files passed shape validation.")

In [ ]:
# STEP 17: Final feature-engineering audit

print("FINAL FEATURE ENGINEERING AUDIT")
print("=" * 65)

key_columns = ["Participant_ID", "Date"]

# 1. Dataset integrity
print("\n1. DATASET INTEGRITY")

print("Rows:", len(final_feature_dataset))
print("Participants:",
      final_feature_dataset["Participant_ID"].nunique())
print("Features:", len(final_feature_dataset.columns) - 2)
print("Duplicate participant-date keys:",
      final_feature_dataset.duplicated(key_columns).sum())
print("Completely empty columns:",
      final_feature_dataset.drop(columns=key_columns)
      .columns[
          final_feature_dataset.drop(columns=key_columns).isna().all()
      ].tolist())

numeric_data = final_feature_dataset.select_dtypes(include="number")

print("Infinite numeric values:",
      np.isinf(numeric_data.to_numpy()).sum())

# 2. Metadata completeness
print("\n2. METADATA COMPLETENESS")

dataset_features = set(final_feature_dataset.columns) - set(key_columns)
summary_features = set(feature_summary["Feature"])
dictionary_features = set(feature_dictionary["Feature"])

print("Dataset feature count:", len(dataset_features))
print("Summary feature count:", len(summary_features))
print("Dictionary feature count:", len(dictionary_features))

print("Dataset vs summary match:",
      dataset_features == summary_features)

print("Dataset vs dictionary match:",
      dataset_features == dictionary_features)

# 3. Verify summary missingness
print("\n3. SUMMARY ACCURACY")

actual_missing = (
    final_feature_dataset.drop(columns=key_columns)
    .isna()
    .sum()
)

reported_missing = (
    feature_summary.set_index("Feature")["Missing_Count"]
)

missing_counts_match = actual_missing.equals(
    reported_missing.reindex(actual_missing.index).astype("int64")
)

print("Missing counts match dataset:", missing_counts_match)

# 4. Feature group distribution
print("\n4. FEATURE GROUP DISTRIBUTION")
print(feature_dictionary["Feature_Group"].value_counts())

# 5. Saved-file location
print("\n5. OUTPUT FILES")

for filename in outputs:
    path = correct_output_dir / filename
    print(filename, ":", path.exists())

# Final assertions
assert len(final_feature_dataset) == 13085
assert len(dataset_features) == 1916
assert not final_feature_dataset.duplicated(key_columns).any()
assert not np.isinf(numeric_data.to_numpy()).any()
assert not final_feature_dataset.drop(
    columns=key_columns
).isna().all().any()
assert dataset_features == summary_features == dictionary_features
assert missing_counts_match
assert all(
    (correct_output_dir / filename).exists()
    for filename in outputs
)

print("\nALL FINAL AUDIT ASSERTIONS PASSED")

In [ ]:
# STEP 18: Independent temporal leakage audit

test_feature = "BRI_rf_mean_mean"
test_participant = "GER-003"

participant_daily = (
    final_daily[
        final_daily["Participant_ID"] == test_participant
    ]
    .sort_values("Date")
    .copy()
)

print("TEMPORAL LEAKAGE AUDIT")
print("=" * 65)

# Check rolling means for 3, 7 and 14 days
for days, min_count in [(3, 1), (7, 2), (14, 3)]:

    suffix = f"_rolling_{days}d_mean"
    count_col = f"{test_feature}_rolling_{days}d_count"

    candidates = rolling_features[
        (rolling_features["Participant_ID"] == test_participant)
        & (rolling_features[count_col] >= min_count)
    ]

    if candidates.empty:
        print(f"\n{days}-day rolling window: No eligible test row")
        continue

    test_row = candidates.iloc[-1]
    test_date = test_row["Date"]

    historical = participant_daily[
        (participant_daily["Date"] < test_date)
        & (
            participant_daily["Date"]
            >= test_date - pd.Timedelta(days=days)
        )
    ][test_feature].dropna()

    expected_count = len(historical)
    expected_mean = historical.mean()

    stored_count = test_row[count_col]
    stored_mean = test_row[test_feature + suffix]

    print(f"\n{days}-DAY ROLLING CHECK")
    print("Date:", test_date)
    print("Expected historical count:", expected_count)
    print("Stored historical count:", stored_count)
    print("Count matches:", expected_count == stored_count)
    print("Expected mean:", expected_mean)
    print("Stored mean:", stored_mean)
    print(
        "Mean matches:",
        np.isclose(expected_mean, stored_mean, equal_nan=True)
    )

# Check expanding historical baseline
baseline_candidates = baseline_features[
    (baseline_features["Participant_ID"] == test_participant)
    & (baseline_features[test_feature + "_baseline_count"] >= 3)
]

print("\nEXPANDING BASELINE CHECK")

if not baseline_candidates.empty:

    baseline_row = baseline_candidates.iloc[-1]
    test_date = baseline_row["Date"]

    historical = participant_daily[
        participant_daily["Date"] < test_date
    ][test_feature].dropna()

    print("Date:", test_date)
    print("Expected historical count:", len(historical))
    print(
        "Stored historical count:",
        baseline_row[test_feature + "_baseline_count"]
    )
    print(
        "Mean matches:",
        np.isclose(
            historical.mean(),
            baseline_row[test_feature + "_baseline_mean"],
            equal_nan=True
        )
    )
    print(
        "Median matches:",
        np.isclose(
            historical.median(),
            baseline_row[test_feature + "_baseline_median"],
            equal_nan=True
        )
    )
    print(
        "Standard deviation matches:",
        np.isclose(
            historical.std(),
            baseline_row[test_feature + "_baseline_std"],
            equal_nan=True
        )
    )

else:
    print("No eligible baseline test row found.")

In [ ]:
# STEP 19: Inspect existing visualizations

import json
from pathlib import Path

notebook_path = Path("04_feature_engineering.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook_data = json.load(f)

plot_cells = []

for i, cell in enumerate(notebook_data["cells"]):
    source = "".join(cell.get("source", []))

    if any(term in source.lower() for term in [
        "plt.", "sns.", "plotly", ".plot(", "hist(", "heatmap"
    ]):
        plot_cells.append({
            "Cell": i,
            "Type": cell["cell_type"],
            "Code_Preview": source[:250].replace("\n", " ")
        })

print("VISUALIZATION INSPECTION")
print("=" * 60)
print("Potential visualization cells:", len(plot_cells))

for item in plot_cells:
    print("\nCell:", item["Cell"])
    print("Type:", item["Type"])
    print("Preview:", item["Code_Preview"])

In [ ]:
# STEP 19A: Visualize top 20 features by missingness

import matplotlib.pyplot as plt

missing_plot = (
    feature_summary
    .sort_values("Missing_Percentage", ascending=False)
    .head(20)
    .sort_values("Missing_Percentage")
)

plt.figure(figsize=(12, 8))

plt.barh(
    missing_plot["Feature"],
    missing_plot["Missing_Percentage"]
)

plt.xlabel("Missing values (%)")
plt.ylabel("Feature")
plt.title("Top 20 Features by Missingness")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 19B: Inspect highly missing features

high_missing = (
    feature_summary
    .merge(
        feature_dictionary[
            ["Feature", "Source_Sensor", "Feature_Group"]
        ],
        on="Feature",
        how="left"
    )
    .query("Missing_Percentage >= 95")
    .sort_values("Missing_Percentage", ascending=False)
)

print("HIGH-MISSINGNESS FEATURE ANALYSIS")
print("=" * 70)
print("Features with >=95% missingness:", len(high_missing))

print(
    high_missing[
        ["Feature", "Source_Sensor", "Feature_Group", "Missing_Percentage"]
    ].to_string(index=False)
)

In [ ]:
# STEP 19C: Inspect EI percentage-change features

ei_features = [
    "EI_all_zero_valid_record_count_rolling_3d_pct_change",
    "EI_all_zero_valid_record_count_rolling_7d_pct_change",
    "EI_all_zero_valid_record_count_rolling_14d_pct_change",
    "EI_Lux_min_rolling_3d_pct_change",
    "EI_Lux_min_rolling_7d_pct_change",
    "EI_Lux_min_rolling_14d_pct_change"
]

for feature in ei_features:
    if feature not in final_feature_dataset.columns:
        print(f"\n{feature}: NOT FOUND")
        continue

    values = final_feature_dataset[feature].dropna()

    print(f"\n{feature}")
    print("Non-missing values:", len(values))
    print("Missing percentage:", round(
        final_feature_dataset[feature].isna().mean() * 100, 2
    ))
    print("Sample values:")
    print(values.head(5).to_string(index=False))

In [ ]:
# STEP 19D: Verify percentage-change zero handling

feature = "EI_all_zero_valid_record_count"

check_dates = (
    final_feature_dataset.loc[
        final_feature_dataset[feature + "_rolling_3d_pct_change"].notna(),
        ["Participant_ID", "Date"]
    ]
    .head(10)
)

for _, row in check_dates.iterrows():
    participant = row["Participant_ID"]
    current_date = row["Date"]

    history = final_daily.loc[
        (final_daily["Participant_ID"] == participant) &
        (final_daily["Date"] < current_date) &
        (final_daily["Date"] >= current_date - pd.Timedelta(days=3)),
        ["Date", feature]
    ].dropna().sort_values("Date")

    print("\nParticipant:", participant, "| Date:", current_date)
    print("Historical observations:")
    print(history.to_string(index=False))

In [ ]:
# STEP 19E: Independently validate EI 3-day percentage changes

feature = "EI_all_zero_valid_record_count"
change_col = feature + "_rolling_3d_pct_change"

checks = []

valid_rows = final_feature_dataset.loc[
    final_feature_dataset[change_col].notna(),
    ["Participant_ID", "Date", change_col]
].head(20)

for _, row in valid_rows.iterrows():
    participant = row["Participant_ID"]
    current_date = row["Date"]

    history = final_daily.loc[
        (final_daily["Participant_ID"] == participant) &
        (final_daily["Date"] < current_date) &
        (final_daily["Date"] >= current_date - pd.Timedelta(days=3)),
        ["Date", feature]
    ].dropna().sort_values("Date")

    values = history[feature].tolist()

    if len(values) >= 2 and values[-2] != 0:
        expected = ((values[-1] - values[-2]) / values[-2]) * 100
    else:
        expected = float("nan")

    actual = row[change_col]

    checks.append({
        "Participant": participant,
        "Date": current_date,
        "Previous": values[-2] if len(values) >= 2 else None,
        "Latest": values[-1] if len(values) >= 1 else None,
        "Expected": expected,
        "Actual": actual,
        "Match": (
            pd.notna(expected) and
            abs(expected - actual) < 1e-6
        )
    })

check_df = pd.DataFrame(checks)

print("PERCENTAGE-CHANGE VALIDATION")
print("=" * 65)
print(check_df.to_string(index=False))
print("\nMatching rows:", check_df["Match"].sum(), "/", len(check_df))

In [ ]:
# STEP 20: Compare missingness across feature groups

group_missingness = (
    feature_summary
    .merge(
        feature_dictionary[["Feature", "Feature_Group"]],
        on="Feature",
        how="left"
    )
    .groupby("Feature_Group")["Missing_Percentage"]
    .agg(["mean", "median"])
    .sort_values("mean", ascending=False)
)

print("MISSINGNESS BY FEATURE GROUP")
print("=" * 60)
print(group_missingness.round(2))

group_missingness.plot(
    kind="bar",
    figsize=(10, 5),
    ylabel="Missingness (%)",
    title="Mean and Median Missingness by Feature Group"
)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# STEP 21: Inspect notebook documentation

import json
from pathlib import Path

notebook_path = Path("04_feature_engineering.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook_data = json.load(f)

print("NOTEBOOK DOCUMENTATION AUDIT")
print("=" * 65)

markdown_count = 0

for i, cell in enumerate(notebook_data["cells"]):
    if cell["cell_type"] == "markdown":
        markdown_count += 1
        content = "".join(cell.get("source", [])).strip()

        if content:
            print(f"\nMarkdown Cell {i}:")
            print(content[:700])

print("\nTotal markdown cells:", markdown_count)

In [ ]:
# STEP 21A: List all notebook section headings

for i, cell in enumerate(notebook_data["cells"]):
    if cell["cell_type"] == "markdown":
        content = "".join(cell.get("source", []))

        print(f"\n--- MARKDOWN CELL {i} ---")
        for line in content.splitlines():
            if line.strip().startswith("#"):
                print(line.strip())

In [ ]:
# STEP 21B: Inspect notebook cell sequence

for i, cell in enumerate(notebook_data["cells"]):
    source = "".join(cell.get("source", [])).strip()
    first_line = source.split("\n")[0][:110] if source else "(empty)"

    print(f"{i:3} | {cell['cell_type']:8} | {first_line}")

## Final Feature-Engineering Outputs

All engineered feature datasets and metadata files were saved under:

`data/processed/features/`

The following eight output files were generated:

| File | Description |
|---|---|
| `daily_sensor_features.parquet` | Daily features from the eight sensors |
| `rolling_features.parquet` | Historical 3-day, 7-day and 14-day rolling features |
| `baseline_features.parquet` | Participant-specific historical baseline features |
| `data_quality_features.parquet` | Sensor availability, observation counts and coverage features |
| `cross_sensor_features.parquet` | Exploratory cross-sensor features |
| `final_feature_dataset.parquet` | Integrated feature dataset |
| `feature_summary.csv` | Feature-level descriptive statistics and missingness |
| `feature_dictionary.csv` | Feature definitions and engineering documentation |

### Final Dataset Overview

The integrated feature dataset contains:

- **13,085** participant-date records.
- **74** unique participants.
- **1,916** engineered feature columns.
- **1,918** total columns, including `Participant_ID` and `Date`.

All eight saved outputs were read back and their dimensions were compared against the corresponding in-memory datasets.

The final audit checked duplicate keys, empty columns, infinite values, metadata consistency and saved-file availability.

### Data-Handling Limitations

The feature dataset preserves the original sensor-data availability patterns. Sparse measurements and high missingness remain visible.

The engineered features are intended for downstream exploratory analysis and COPD exacerbation-risk modelling. They do not independently establish clinical diagnoses, causal relationships or validated prediction performance.

In [ ]:
# STEP 22: Final notebook structure audit

import json
from pathlib import Path

notebook_path = Path("04_feature_engineering.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    final_notebook = json.load(f)

print("FINAL NOTEBOOK AUDIT")
print("=" * 65)

markdown_cells = [
    (i, "".join(cell.get("source", [])).strip())
    for i, cell in enumerate(final_notebook["cells"])
    if cell["cell_type"] == "markdown"
]

empty_cells = [
    i for i, cell in enumerate(final_notebook["cells"])
    if not "".join(cell.get("source", [])).strip()
]

print("Total cells:", len(final_notebook["cells"]))
print("Markdown cells:", len(markdown_cells))
print("Code cells:", sum(
    cell["cell_type"] == "code"
    for cell in final_notebook["cells"]
))
print("Empty cells:", empty_cells)

print("\nDOCUMENTATION HEADINGS")
for i, content in markdown_cells:
    for line in content.splitlines():
        if line.strip().startswith("#"):
            print(f"Cell {i}: {line.strip()}")

print("\nDOCUMENTATION CHECKS")
required_sections = [
    "Sampling-Frequency and Data-Coverage Analysis",
    "Common Participant-Date Timeline",
    "Daily Sensor Feature Engineering",
    "Unified Daily Feature Dataset",
    "Feature Integration and Temporal Validation",
    "Temporal Leakage Checks",
    "Missingness and Numerical Validation",
    "Feature Summary and Dictionary",
    "Final Feature-Engineering Outputs",
    "Final Dataset Overview",
    "Data-Handling Limitations"
]

all_markdown = "\n".join(content for _, content in markdown_cells)

for section in required_sections:
    found = section.lower() in all_markdown.lower()
    print(f"{'PASS' if found else 'MISSING'}: {section}")

In [ ]:
# STEP 22A: Inspect documentation insertion locations

for i in list(range(75, 82)) + list(range(
    max(0, len(final_notebook["cells"]) - 6),
    len(final_notebook["cells"])
)):
    cell = final_notebook["cells"][i]
    source = "".join(cell.get("source", [])).strip()

    print(f"\nCell {i} | Type: {cell['cell_type']}")
    print(source[:500] if source else "[EMPTY]")

In [ ]:
# Inspect all percentage-change formulas in the feature dictionary

pct_change_dictionary = feature_dictionary[
    feature_dictionary["Feature"].str.endswith("_pct_change")
]

print("Percentage-change entries:", len(pct_change_dictionary))

print("\nFormula distribution:")
print(pct_change_dictionary["Formula"].value_counts())

print("\nSample percentage-change features:")
display(
    pct_change_dictionary[
        ["Feature", "Description", "Formula"]
    ].head(15)
)